<a href="https://colab.research.google.com/github/kimdonggyu2008/Personal_Study/blob/main/wav2vec_%EA%B5%AC%ED%98%84.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 사전설정

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# %cd /content/drive/MyDrive/코딩공부/project_folder/fairseq
!pip install git+https://github.com/One-sixth/fairseq.git

  Cloning https://github.com/One-sixth/fairseq.git to /tmp/pip-req-build-9q1oh47i
  Running command git clone --filter=blob:none --quiet https://github.com/One-sixth/fairseq.git /tmp/pip-req-build-9q1oh47i
  Resolved https://github.com/One-sixth/fairseq.git to commit 44800430a728c2216fd1cf1e8daa672f50dfacba
  Running command git submodule update --init --recursive -q
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [3]:
# !pip install pip==23.3.1


In [4]:
# !pip install --editable ./


In [5]:
# # ✅ 1. Python 3.9 설치
# !sudo apt-get update -y
# !sudo apt-get install python3.9 python3.9-dev python3.9-distutils -y

# # ✅ 2. Python 3.9용 pip 설치
# !wget https://bootstrap.pypa.io/get-pip.py
# !python3.9 get-pip.py

# # ✅ 3. 가상환경 생성 (선택적)
# !python3.9 -m venv py39env

# # ✅ 4. Colab에 Python 3.9 커널 연결
# !py39env/bin/pip install ipykernel
# !py39env/bin/python -m ipykernel install --user --name python39 --display-name "Python 3.9 (Custom)"


# utils.py

In [6]:
!pip install hydra-core==1.3.2 omegaconf==2.3.0

In [7]:
!pip install torch==1.13.1 torchaudio==0.13.1 \
  numpy==1.24.4 \
  extension_helpers \
  jinja2 astropy scipy==1.9.3 \
  editdistance pyctcdecode librosa==0.9.2 \
  tqdm pyyaml cython setuptools \
  git+https://github.com/kpu/kenlm.git


  Cloning https://github.com/kpu/kenlm.git to /tmp/pip-req-build-lp14rzr2
  Running command git clone --filter=blob:none --quiet https://github.com/kpu/kenlm.git /tmp/pip-req-build-lp14rzr2
  Resolved https://github.com/kpu/kenlm.git to commit 4cb443e60b7bf2c0ddf3c745378f76cb59e254e5
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached torch-1.13.1-cp311-cp311-manylinux1_x86_64.whl.metadata (24 kB)
ERROR: Ignored the following yanked versions: 2.0.0
ERROR: Could not find a version that satisfies the requirement torchaudio==0.13.1 (from versions: 2.0.1, 2.0.2, 2.1.0, 2.1.1, 2.1.2, 2.2.0, 2.2.1, 2.2.2, 2.3.0, 2.3.1, 2.4.0, 2.4.1, 2.5.0, 2.5.1, 2.6.0, 2.7.0)
ERROR: No matching distribution found for torchaudio==0.13.1


In [8]:
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import argparse
import collections
import contextlib
import copy
import logging
import os
import sys
import warnings
import time
import re
import librosa
import json
from omegaconf import II
import wave
import unicodedata
import random
from tqdm import tqdm
import yaml

from argparse import Namespace
from omegaconf import DictConfig, OmegaConf
from torch import Tensor
from itertools import accumulate, chain
from dataclasses import dataclass, field
from typing import Any, List, Tuple, TYPE_CHECKING, Callable, Dict, Optional
from sklearn.model_selection import train_test_split
from pathlib import Path
import soundfile as sf
from preprocess import preprocess

In [9]:
# 추가되면 지우기


from fairseq import utils
from fairseq.data.data_utils import compute_mask_indices
from fairseq.dataclass import ChoiceEnum, FairseqDataclass
from fairseq.distributed import fsdp_wrap
from fairseq.models import BaseFairseqModel, register_model
from fairseq.distributed.fully_sharded_data_parallel import FullyShardedDataParallel
from fairseq.modules import (
    Fp32GroupNorm,
    Fp32LayerNorm,
    GradMultiply,
    GumbelVectorQuantizer,
    LayerNorm,
    MultiheadAttention,
    RelPositionalEncoding,
    SamePad,
    TransposeLast,
)
from fairseq.modules.checkpoint_activations import checkpoint_wrapper
from fairseq.modules.conformer_layer import ConformerWav2Vec2EncoderLayer
from fairseq.modules.transformer_sentence_encoder import init_bert_params
from fairseq.utils import buffered_arange, index_put, is_xla_tensor
from fairseq import checkpoint_utils, options, quantization_utils, tasks, utils
from fairseq.data import data_utils, iterators
from fairseq.data.plasma_utils import PlasmaStore
from fairseq.dataclass.configs import FairseqConfig
from fairseq.dataclass.initialize import add_defaults
from fairseq.dataclass.utils import convert_namespace_to_omegaconf
from fairseq.distributed import fsdp_enable_wrap, fsdp_wrap
from fairseq.distributed import utils as distributed_utils
from fairseq.file_io import PathManager
from fairseq.logging import meters, metrics, progress_bar
from fairseq.model_parallel.megatron_trainer import MegatronTrainer
#from fairseq.trainer import Trainer

from fairseq import checkpoint_utils, distributed_utils, models, optim, utils
from fairseq.file_io import PathManager
from fairseq.nan_detector import NanDetector
from fairseq.optim import lr_scheduler



from fairseq import metrics, utils
from fairseq.criterions import FairseqCriterion, register_criterion
from fairseq.dataclass import FairseqDataclass
from fairseq.data.data_utils import post_process
from fairseq.tasks import FairseqTask
from fairseq.logging.meters import safe_round

#from .utils import pad_to_multiple

In [10]:


# We need to setup root logger before importing any fairseq libraries.
logging.basicConfig(
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    level=os.environ.get("LOGLEVEL", "INFO").upper(),
    stream=sys.stdout,
)
logger = logging.getLogger("fairseq_cli.train")






In [11]:
EXTRACTOR_MODE_CHOICES=ChoiceEnum(["default","layer_norm"])
MASKING_DISTRIBUTION_CHOICES=ChoiceEnum(["static","uniform","normal","poisson"])

In [12]:
@dataclass
class Wav2Vec2Config(FairseqDataclass):
    extractor_mode: EXTRACTOR_MODE_CHOICES = field(
        default="default",
        metadata={
            "help": "mode for feature extractor. default has a single group norm with d "
            "groups in the first conv block, whereas layer_norm has layer norms in "
            "every block (meant to use with normalize=True)"
        },
    )
    encoder_layers: int = field(
        default=12, metadata={"help": "num encoder layers in the transformer"}
    )
    encoder_embed_dim: int = field(
        default=768, metadata={"help": "encoder embedding dimension"}
    )
    encoder_ffn_embed_dim: int = field(
        default=3072, metadata={"help": "encoder embedding dimension for FFN"}
    )
    encoder_attention_heads: int = field(
        default=12, metadata={"help": "num encoder attention heads"}
    )
    activation_fn: ChoiceEnum(utils.get_available_activation_fns()) = field(
        default="gelu", metadata={"help": "activation function to use"}
    )

    # dropouts
    dropout: float = field(
        default=0.1, metadata={"help": "dropout probability for the transformer"}
    )
    attention_dropout: float = field(
        default=0.1, metadata={"help": "dropout probability for attention weights"}
    )
    activation_dropout: float = field(
        default=0.0, metadata={"help": "dropout probability after activation in FFN"}
    )
    encoder_layerdrop: float = field(
        default=0.0, metadata={"help": "probability of dropping a tarnsformer layer"}
    )
    dropout_input: float = field(
        default=0.0,
        metadata={"help": "dropout to apply to the input (after feat extr)"},
    )
    dropout_features: float = field(
        default=0.0,
        metadata={"help": "dropout to apply to the features (after feat extr)"},
    )

    final_dim: int = field(
        default=0,
        metadata={
            "help": "project final representations and targets to this many dimensions."
            "set to encoder_embed_dim is <= 0"
        },
    )
    layer_norm_first: bool = field(
        default=False, metadata={"help": "apply layernorm first in the transformer"}
    )
    conv_feature_layers: str = field(
        default="[(512, 10, 5)] + [(512, 3, 2)] * 4 + [(512,2,2)] + [(512,2,2)]",
        metadata={
            "help": "string describing convolutional feature extraction layers in form of a python list that contains "
            "[(dim, kernel_size, stride), ...]"
        },
    )
    conv_bias: bool = field(
        default=False, metadata={"help": "include bias in conv encoder"}
    )
    logit_temp: float = field(
        default=0.1, metadata={"help": "temperature to divide logits by"}
    )
    quantize_targets: bool = field(
        default=False, metadata={"help": "use quantized targets"}
    )
    quantize_input: bool = field(
        default=False, metadata={"help": "use quantized inputs"}
    )
    same_quantizer: bool = field(
        default=False, metadata={"help": "use same quantizer for inputs and targets"}
    )
    target_glu: bool = field(
        default=False, metadata={"help": "adds projection + glu to targets"}
    )
    feature_grad_mult: float = field(
        default=1.0, metadata={"help": "multiply feature extractor var grads by this"}
    )
    latent_vars: int = field(
        default=320,
        metadata={"help": "number of latent variables V in each group of the codebook"},
    )
    latent_groups: int = field(
        default=2,
        metadata={"help": "number of groups G of latent variables in the codebook"},
    )
    latent_dim: int = field(
        default=0,
        metadata={
            "help": "if > 0, uses this dimensionality for latent variables. "
            "otherwise uses final_dim / latent_groups"
        },
    )

    # masking
    mask_length: int = field(default=10, metadata={"help": "mask length"})
    mask_prob: float = field(
        default=0.65, metadata={"help": "probability of replacing a token with mask"}
    )
    mask_selection: MASKING_DISTRIBUTION_CHOICES = field(
        default="static", metadata={"help": "how to choose mask length"}
    )
    mask_other: float = field(
        default=0,
        metadata={
            "help": "secondary mask argument (used for more complex distributions), "
            "see help in compute_mask_indices"
        },
    )
    no_mask_overlap: bool = field(
        default=False, metadata={"help": "whether to allow masks to overlap"}
    )
    mask_min_space: int = field(
        default=1,
        metadata={"help": "min space between spans (if no overlap is enabled)"},
    )

    # channel masking
    mask_channel_length: int = field(
        default=10, metadata={"help": "length of the mask for features (channels)"}
    )
    mask_channel_prob: float = field(
        default=0.0, metadata={"help": "probability of replacing a feature with 0"}
    )
    mask_channel_selection: MASKING_DISTRIBUTION_CHOICES = field(
        default="static",
        metadata={"help": "how to choose mask length for channel masking"},
    )
    mask_channel_other: float = field(
        default=0,
        metadata={
            "help": "secondary mask argument (used for more complex distributions), "
            "see help in compute_mask_indicesh"
        },
    )
    no_mask_channel_overlap: bool = field(
        default=False, metadata={"help": "whether to allow channel masks to overlap"}
    )
    mask_channel_min_space: int = field(
        default=1,
        metadata={"help": "min space between spans (if no overlap is enabled)"},
    )

    # negative selection
    num_negatives: int = field(
        default=100,
        metadata={"help": "number of negative examples from the same sample"},
    )
    negatives_from_everywhere: bool = field(
        default=False,
        metadata={"help": "sample negatives from everywhere, not just masked states"},
    )
    cross_sample_negatives: int = field(
        default=0, metadata={"help": "number of negative examples from the any sample"}
    )
    codebook_negatives: int = field(
        default=0, metadata={"help": "number of negative examples codebook"}
    )

    # positional embeddings
    conv_pos: int = field(
        default=128,
        metadata={"help": "number of filters for convolutional positional embeddings"},
    )
    conv_pos_groups: int = field(
        default=16,
        metadata={"help": "number of groups for convolutional positional embedding"},
    )

    latent_temp: Tuple[float, float, float] = field(
        default=(2, 0.5, 0.999995),
        metadata={
            "help": "temperature for latent variable sampling. "
            "can be tuple of 3 values (start, end, decay)"
        },
    )


In [13]:
@register_model("wav2vec2",dataclass=Wav2Vec2Config)
class Wav2Vec2Model(BaseFairseqModel): # 모델 구축
  def __init__(self,cfg: Wav2Vec2Config):
    super().__init__()
    self.cfg=cfg

    feature_enc_layers=eval(cfg.conv_feature)

    self.feature_extractor=ConvFeatureExtractionModel( #cnn기반 low-level 특징 추출
        conv_layers=feature_enc_layers,
        dropout=0.0,
        mode=cfg.extractor_mode,
        conv_bias=cfg.conv_bias,
    )

    self.post_extract_proj=( #feature extraction의 차원이 encoder 차원과 다르면 projection 수행
        nn.Linear(self.embed,cfg.encoder_embed_dim)
        if self.embed!=cfg.encoder_embed_dim and not cfg.quantize_input
        else None
    )

    self.mask_prob = cfg.mask_prob
    self.mask_selection = cfg.mask_selection
    self.mask_other = cfg.mask_other
    self.mask_length = cfg.mask_length
    self.no_mask_overlap = cfg.no_mask_overlap
    self.mask_min_space = cfg.mask_min_space

    self.mask_channel_prob = cfg.mask_channel_prob
    self.mask_channel_selection = cfg.mask_channel_selection
    self.mask_channel_other = cfg.mask_channel_other
    self.mask_channel_length = cfg.mask_channel_length
    self.no_mask_channel_overlap = cfg.no_mask_channel_overlap
    self.mask_channel_min_space = cfg.mask_channel_min_space

    self.dropout_input = nn.Dropout(cfg.dropout_input)
    self.dropout_features = nn.Dropout(cfg.dropout_features)

    self.feature_grad_mult = cfg.feature_grad_mult

    self.quantizer = None
    self.input_quantizer = None

    self.n_negatives = cfg.num_negatives
    self.cross_sample_negatives = cfg.cross_sample_negatives
    self.codebook_negatives = cfg.codebook_negatives
    self.negatives_from_everywhere = cfg.negatives_from_everywhere

    self.logit_temp = cfg.logit_temp

    final_dim=cfg.final_dim if cfg.final_dim > 0 else cfg.encoder_embed_dim

    if cfg.quantize_targets:
      vq_dim=cfg.latent_dim if cfg.latent_dim > 0 else final_dim
      self.quantizer=GumbelVectorQuantizer( #gumbel softmax 사용
          dim=self.embed,
          num_vars=cfg.latent_vars,
          temp=cfg.latent_temp,
          combine_groups=False,
          vq_dim=vq_dim,
          time_first=True,
      )
      self.project_q=nn.Linear(vq_dim,final_dim)
    else:
      self.project_q=nn.Linear(self.embed, final_dim)

    if cfg.quantize_input:
      if cfg.same_quantizer and self.quantizer is not None:
        self.input_quantizer=self.quantizer

      else:
        vq_dim=cfg.latent_dim if cfg.latent_dim>0 else cfg.encoder_embed_dim
        self.input_quantizer=GumbelVectorQuantizer(
            dim=self.embed,
            num_vars=cfg.latent_vars,
            temp=cfg.latent_temp,
            groupts=cfg.latent_groups,
            combine_groups=False,
            vq_dim=vq_dim,
            time_first=True,
        )
      self.project_inp=nn.Linear(vq_dim,cfg.encoder_embed_dim)

    self.mask_emb=nn.Parameter(torch.FloatTensor(cfg.encoder_embed_dim).uniform_())
    self.encoder=TransformerEncoder(cfg)
    self.layer_norm=LayerNorm(self.embed)

    self.target_glu=None
    if cfg.target_glu:
      self.target_glu=nn.Sequential(nn.Linear(final_dim,final_dim*2),nn.GLU())
    self.final_proj=nn.Linear(cfg.encoder_embed_dim,final_dim)

  def upgrade_state_dict_named(self,state_dict,name):
    super().upgrade_state_dict_named(state_dict,name)
    return state_dict

  @classmethod
  def build_model(cls,cfg:Wav2Vec2Config,task=None):
    return cls(cfg)

  def apply_mask(self,x,padding_mask): #마스크 적용
    B,T,C=x.shape # (b,t,c)
    if self.mask_prob>0:
      mask_indices=compute_mask_indices( #시간 도메인 마스크에 대한 조건 지정
          (B,T),
          padding_mask,
          self.mask_prob,
          self.mask_length,
          self.mask_selection,
          self.mask_other,
          min_mask=2,
          no_overlap=self.no_mask_overlap,
          min_space=self.mask_min_space,
      )

      mask_indices=torch.from_numpy(mask_indices).to(x.device)
      x[mask_indices]=self.mask_emb

    else:
      mask_indices=None

    if self.mask_channel_prob>0: #마스크
      mask_channel_indices=compute_mask_indices( #채널 도메인 마스크 조건 지정
          (B,C),
          None,
          self.mask_channel_prob,
          self.mask_channel_length,
          self.mask_channel_selection,
          self.mask_channel_other,
          no_overlap=self.no_mask_channel_overlap,
          min_space=self.mask_channel_min_space,
      )
      mask_channel_indices=(
          torch.from_numpy(mask_channel_indices).to(x.device.unsqueeze(1).expand(-1,T,-1))
      )
      x[mask_channel_indices]=0

    return x,mask_indices

  def sample_negatives(self,y,num): #정답 예제와 비교할 가짜예제 샘플링

    # negative 샘플링 수가 0이면 빈 tensor 반환
    if self.n_negatives==0 and self.cross_sample_negatives==0:
      return y.new(0)

    # (b,t,f)
    bsz,tsz,fsz=y.shape
    #(b*t,f)
    y=y.view(-1,fsz)

    # 다른 샘플에서 가져올 인덱스를 랜덤하게 샘플링
    cross_high=tsz*bsz
    high=tsz
    with torch.no_grad():
      assert high >1, f"{bsz,tsz,fsz}"

      if self.n_negatives>0:
        tszs=(
            buffered_arange(num).unsqueeze(-1).expand(-1,self.cross_sample_negatives).flatten()
        )
        cross_neg_idxs=torch.randint(low=0,high=cross_high-1, size=(bsz,self.cross_sample_negatives*num),)
        cross_neg_idxs[cross_neg_idxs>=tszs]+=1

    # 동일 샘플 negative 샘플링
    if self.n_negatives>0:
      for i in range(1,bsz):
        neg_idxs[i]+=i*high
    else:
      neg_idxs=cross_neg_idxs

    # cross, same negative 합치기
    if self.cross_sample_negatives>0 and self.n_negatives>0:
      neg_idxs=torch.cat([neg_idxs, cross_neg_idxs], dim=1)

    # 샘플된 인덱스로 negative 벡터 추출 및 정형화
    negs=y[neg_idxs.view(-1)]
    negs=negs.view(bsz,num,self.n_negatives+self.cross_sample_negatives,fsz).permute(2,0,1,3)
    return negs, neg_idxs


  def compute_preds(self,x,y,negatives):  #contrative loss
    # 각각의 예측값과 정답값을 비교해 손실 찾아냄
    neg_is_pos=(y==negatives).all(-1)
    y=y.unsqueeze(0)
    targets=torch.cat([y,negatives],dim=0)

    logits=torch.cosine_similarity(x.float(),targets.float().type_as(x))

    logits/=self.logit_temp

    if neg_is_pos.any():
      logits[1:][neg_is_pos]=float("-inf")

    return logits


  # 진행
  def forward(self,source,padding_mask=None, mask=True,features_only=False):

    if self.feature_grad_mult>0:
      features=self.feature_extractor(source)

      if self.feature_grad_mult!=1.0:# feature gradient 조절
        features=GradMultiply.apply(features, self.feature_grad_mult)
      else:
        with torch.no_grad():
          features=self.feature_extractor(source)

      features_pen=features.float().pow(2).mean()

      features=features.transpose(1,2) #(b,t,c)
      features=self.layer_norm(features) #정규화
      unmasked_features=features.clone() #원본 복사

      if padding_mask is not None: #패딩 마스킹 처리
        extra=padding_mask.size(1)%features.size(1)

        if extra >0:
          padding_mask=padding_mask.view(padding_mask.size(0),features.size(1),-1)
          padding_mask=padding_mask.all(-1)


      if self.post_extract_proj is not None: #projection
        features=self.post_extract_proj(features)

      features=self.dropout_input(features) #드롭아웃
      unmasked_features=self.dropout_features(unmasked_features)

      num_vars=None
      code_ppl=None
      prob_ppl=None
      curr_temp=None

      if self.input_quantizer: #입력 양자화
        q=self.input_quantizer(features,produce_targets=False)
        feature=q["x"]
        num_vars=q["num_vars"]
        code_ppl=q["code_perplexity"]
        prob_ppl=q["prob_perplexity"]
        curr_temp=q["temp"]
        features=self.project_inp(features)

      if mask: #마스크 지정
        x, mask_indices = self.apply_mask(features, padding_mask) #마스크 적용
        if mask_indices is not None:
          y = unmasked_features[mask_indices].view(
              unmasked_features.size(0), -1, unmasked_features.size(-1)
          )
        else:
          y = unmasked_features

      else:
        x=features
        y=unmasked_features
        mask_indices=None

      # 인코더 사용, 트랜스포머 계층
      x=self.encoder(x,paddig_mask=padding_mask)

      if features_only:
        return {"x": x, "padding_mask":padding_mask}

      if self.quantizer:
        q=self.quantizer(y,produce_targets=False)
        y=q["x"]
        num_vars=q["num_vars"]
        code_ppl=q["code_perplexity"]
        prob_ppl=q["prob_perplexity"]
        curr_temp=q["temp"]

        y=self.project_q(y) #프로젝션

        if self.negatives_from_everywhere: #negative 샘플링, 의도적으로 정답이 아닌 예시를 같이 보여줌
          neg_cands, *_=self.quantizer(unmasked_features,produce_targets=False)
          negs,_=self.sample_negatives(neg_cands,y.size(1))
          negs=self.project_q(negs)

        else:
          negs,_=self.sample_negatives(y,y.size(1))


        if self.codebook_negatives>0: #코드북에서의 negative 샘플
          cb_negs=self.quantizer.sample_from_codebook(y.size(0)*y.size(1),self.codebook_negatives)
          cb_negs=cb_negs.view(self.codebook_negatives, y.size(0),y.size(1),-1)
          cb_negs=self.project_q(cb_negs)
          negs=torch.cat([negs,cb_negs],dim=0)
        else:
          y=self.project_q(y)

          if self.negatives_from_everywhere: #negative
            negs,_=self.sample_negatives(unmasked_features, y.size(1))
            negs=self.project_q(negs)
          else:
            negs,_=self.sample_negatives(y,y.size(1))

        # 마스킹된 위치의 인코더 출력만 추출
        x=x[mask_indices].view(x.size(0),-1,x.size(-1))

        # glu 적용
        if self.target_glu:
          y=self.target_glu(y)
          negs=self.target_glu(negs)

        x=self.final_proj(x)
        x=self.compute_preds(x,y,negs)

        # 추가 정보 수집 후 결과 반환
        result = {"x": x, "padding_mask": padding_mask, "features_pen": features_pen}

        if prob_ppl is not None:
          result["prob_perplexity"]=prob_ppl
          result["code_perplexity"]=code_ppl
          result["num_vars"]=num_vars
          result["temp"]=curr_temp

        return result


  def quantize(self,x): #입력 음성을 feature로 변환, 양자화 시킨 index로 변경
    assert self.quantizer is not None
    x=self.feature_extractor(x)
    x=x.transpose(1,2)
    x=self.layer_norm(x)
    return self.quantizer.forward_idx(x)

  def extract_features(self,source,padding_mask,mask=False): #forward 사용, 대신 feature만 추출
    res=self.forward(source,padding_mask,mask=mask,features_only=True)
    return res["x"],res["padding_mask"]

  def get_logits(self, net_output):#contrastive loss 계산을 위한 logits 형상 정리
    logits=net_output["x"]
    logits=logits.transpose(0,2)
    logits=logits.reshape(-1,logits.size(-1))
    return logits

  def get_targets(self,sample,net_output,expand_steps=True):
    x=net_output["x"]
    return x.new_zeros(x.size(1)*x.size(2),dtype=torch.long)

  def get_extra_losses(self, net_output): #크로스 엔트로피를 위한 target 벡터 생성
    pen = []

    if "prob_perplexity" in net_output:
      pen.append(
          (net_output["num_vars"] - net_output["prob_perplexity"])
          / net_output["num_vars"]
      )

    if "features_pen" in net_output:
      pen.append(net_output["features_pen"])

    return pen

  def remove_pretraining_modules(self):
    self.quantizer=None
    self.project_q=None
    self.target_glu=None
    self.final_proj=None

In [14]:
class ConvFeatureExtractionModel(nn.Module): #cnn 추출 모델

  def __init__(
    self,
    conv_layers: List[Tuple[int, int, int]],
    dropout: float = 0.0,
    mode: str = "default",
    conv_bias: bool = False,
  ):
    super().__init__()

    assert mode in {"default","layer_norm"}

  def block( #블록
      n_in,
      n_out,
      k,
      stride,
      is_layer_norm=False,
      is_group_norm=False,
      conv_bias=False,
  ):

    def make_conv(): #conv 만들기
      conv = nn.Conv1d(n_in, n_out, k, stride=stride, bias=conv_bias)
      nn.init.kaiming_normal_(conv.weight)
      return conv

    assert (
                is_layer_norm and is_group_norm
            ) == False, "layer norm and group norm are exclusive"

    if is_layer_norm:
      return nn.Sequential( #layer norm 블럭
          make_conv(),
          nn.Dropout(p=dropout),
          nn.Sequential(
              TransposeLast(),
              Fp32LayerNorm(dim, elementwise_affine=True),
              TransposeLast(),
          ),
          nn.GELU(),
      )
    elif is_group_norm:
      return nn.Sequential(
          make_conv(),
          nn.Dropout(p=dropout),
          Fp32GroupNorm(dim,dim,affine=True),
          nn.GELU(),
      )
    else:
      return nn.Sequential(make_conv(),nn.Dropout(p=dropout),nn.GELU())

    in_d=1

    self.conv_layers=nn.ModuleList() #레이어 쌓기
    for i, cl in enumerate(conv_layers):
      assert len(cl)==3, "invalid conv definition: " + str(cl)
      (dim,k,stride)=cl

      self.conv_layers.append(
          block(
              in_d,
              dim,
              k,
              stride,
              is_layer_norm=mode=="layer_norm",
              is_group_norm=mode=="default" and i ==0,
              conv_bias=conv_bias,
          )
      )
      in_d = dim

  def forward(self,x):
    x=x.unsqueeze(1)
    for conv in self.conv_layers:
      x=conv(x)
    return x

In [15]:
class TransformerEncoder(nn.Module): #문맥정보 추출
  def __init__(self,args):
    super().__init__()

    self.dropout=args.dropout
    self.embedding_dim=args.encoder_embed_dim

    self.pos_conv=nn.Conv1d( #후처리용 레이어, positional convolution
        self.embedding_dim,
        self.embedding_dim,
        kernel_size=args.conv_pos,
        padding=args.conv_pos//2,
        groups=args.conv_pos_groups,
    )
    dropout=0

    std=math.sqrt((4*(1.0-dropout))/(args.conv_pos * self.embedding_dim))
    nn.init.normal_(self.pos_conv.weight,mean=0,std=std)
    nn.init.constant_(self.pos_conv.vias,0)

    self.pos_conv=nn.utils.weight_norm(self.pos_conv,name="weight",dim=2)
    self.pos_conv=nn.Sequential(self.pos_conv,SamePad(args.conv_pos),nn.GELU())

    self.layers=nn.ModuleList(
        [ #multi head self attention
          #position wised feed forward network
          # 드롭아웃, residual, norm
            TransformerSentenceEncoderLayer(
                embedding_dim=self.embedding_dim,
                ffn_embedding_dim=args.encoder_ffn_embed_dim,
                num_attention_heads=args.encoder_attention_heads,
                dropout=args.dropout,
                attention_dropout=args.attention_dropout,
                activation_dropout=args.activation_dropout,
                activation_fn=args.activation_fn,
                layer_norm_first=args.layer_norm_first,
            )
            for _ in range(args.encoder_layers)
        ]
    )

    self.layer_norm_first=args.layer_norm_first
    self.layer_norm=LayerNorm(self.embedding_dim)
    self.layerdrop=args.encoder_layerdrop

    self.apply(init_bert_params)


  def forward(self,x,padding_mask=None):
    x=self.extract_features(x,padding_mask) #특징 추출

    if self.layer_norm_first:
      x=self.layer_norm(x)

    return x


  def extract_features(self,x,padding_mask=None):
    if padding_mask is not None:
      x[padding_mask]=0

    x_conv=self.pos_conv(x.transpose(1,2))
    x_conv=x_conv.transpose(1,2)
    x+=x_conv

    if not self.layer_norm_first:
      x=self.layer_norm(x)

    x=F.dropout(x,p=self.dropout, training=self.training)

    x=x.transpose(0,1)

    layer_results=[]

    for i, layer in enumerate(self.layers):
      dropout_probability=np.random.random()
      if not self.training or (dropout_probability > self.layerdrop):
        x,z = layer(x, self_attn_padding_mask=padding_mask, need_weights=False)
        layer_results.append(x)

    x=x.transpose(0,1)

    return x


  def max_position(self):
    return self.args.max_positions

  def upgrade_state_dict_named(self,state_dict,name):
    return state_dict




In [16]:
class TransformerSentenceEncoderLayer(nn.Module):
  def __init__( #bert 스타일 transformer 인코더 레이어 구성
      self,
      embedding_dim: float=768,
      ffn_embedding_dim: float=3072,
      num_attention_heads:float=8,
      dropout: float=0.1,
      attention_dropout: float=0.1,
      activation_dropout: float=0.1,
      activation_fn:str="relu",
      layer_norm_first: bool=False,
  ) -> None:
    super().__init__()

    self.embedding_dim=embedding_dim
    self.dropout=dropout
    self.activation_dropout=activation_dropout

    self.activation_fn=utils.get_activation_fn(activation_fn)
    self.self_attn=MultiheadAttention(
        self.embedding_dim,
        num_attention_heads,
        dropout=attention_dropout,
        self_attention=True,
        )

    self.dropout1=nn.Dropout(dropout)
    self.dropout2=nn.Dropout(self.activation_dropout)
    self.dropout3=nn.Dropout(dropout)

    self.layer_norm_first=layer_norm_first

    self.self_attn_layer_norm=LayerNorm(self.embedding_dim)
    self.fc1=nn.Linear(self.embedding_dim, ffn_embedding_dim)
    self.fc2=nn.Linear(self.embedidng_dim, self.embedding_dim)

    self.final_layer_norm=LayerNorm(self.embedding_dim)


  def forward(
      self,
      x: torch.Tensor,
      self_attn_mask:torch.Tensor=None,
      self_attn_padding_mask: torch.Tensor=None,
      need_weights: bool=False,
      att_args=None,
  ):
    residual=x #잔차 적용

    if self.layer_norm_first:
      x=self.self_attn_layer_norm(x)
      x,attn=self.self_attn(#셀프 어텐션
          query=x,
          key=x,
          value=x,
          key_padding_mask=self_attn_padding_mask,
          need_weights=False,
          attn_mask=self_attn_mask,
      )


      x=self.dropout1(x) #드롭아웃
      x=residual+x #잔차 적용

      residual=x

      x=self.final_layer_norm(x)
      x=self.activation_fn(self,fc1(x))
      x=self.dropout2(x)
      x=self.fc2(x)
      x=self.dropout3(x)
      x=residual+x

    else:
      x,attn=self.self_attn(
          query=x,
          key=x,
          value=x,
          key_padding_mask=self_attn_padding_mask,
          need_weights=need_weights,
      )

      x=self.dropout1(x)

      x=residual+x

      x=self.self_attn_layer_norm(x)

      residual=x

      x=self.activation_fn(self.fc1(x))
      x=self.dropout2(x)
      x=self.fc2(x)
      x=self.dropout3(x)
      x=residual+x
      x=self.final_layer_norm(x)

    return x,attn




#criterion.py

In [17]:
@dataclass
class CtcCriterionConfig(FairseqDataclass): #설정값 정의
    zero_infinity: bool = field( #ctc 계산시 log0 에러 방지용
        default=False,
        metadata={"help": "zero inf loss when source length <= target length"},
    )
    sentence_avg: bool = II("optimization.sentence_avg") #loss 평균을 문장 기준 or true 토큰 기준
    post_process: str = field(# 디코딩 후 처리방식
        default="letter",
        metadata={
            "help": "how to post process predictions into words. can be letter, "
            "wordpiece, BPE symbols, etc. "
            "See fairseq.data.data_utils.post_process() for full list of options"
        },
    )
    wer_kenlm_model: Optional[str] = field(
        default=None,
        metadata={
            "help": "if this is provided, use kenlm to compute wer (along with other wer_* args)"
        },
    )
    wer_lexicon: Optional[str] = field(
        default=None,
        metadata={"help": "lexicon to use with wer_kenlm_model"},
    )
    wer_lm_weight: float = field(
        default=2.0,
        metadata={"help": "lm weight to use with wer_kenlm_model"},
    )
    wer_word_score: float = field(
        default=-1.0,
        metadata={"help": "lm word score to use with wer_kenlm_model"},
    )

    wer_args: Optional[str] = field(
        default=None,
        metadata={
            "help": "DEPRECATED: tuple of (wer_kenlm_model, wer_lexicon, wer_lm_weight, wer_word_score)"
        },
    )



In [20]:

#@register_criterion("ctc", dataclass=CtcCriterionConfig)
class CtcCriterion(FairseqCriterion):
    def __init__(self, cfg: CtcCriterionConfig, task: FairseqTask):
        super().__init__(task)
        self.blank_idx = task.target_dictionary.index(task.blank_symbol) if hasattr(task, 'blank_symbol') else 0
        self.pad_idx = task.target_dictionary.pad() #각각의 추가 토큰 인덱스 지정
        self.eos_idx = task.target_dictionary.eos()
        self.post_process = cfg.post_process #후처리

        if cfg.wer_args is not None:
            (
                cfg.wer_kenlm_model,
                cfg.wer_lexicon,
                cfg.wer_lm_weight,
                cfg.wer_word_score,
            ) = eval(cfg.wer_args)

        if cfg.wer_kenlm_model is not None:
            from examples.speech_recognition.w2l_decoder import W2lKenLMDecoder

            dec_args = Namespace()
            dec_args.nbest = 1
            dec_args.criterion = "ctc"
            dec_args.kenlm_model = cfg.wer_kenlm_model
            dec_args.lexicon = cfg.wer_lexicon
            dec_args.beam = 50
            dec_args.beam_size_token = min(50, len(task.target_dictionary))
            dec_args.beam_threshold = min(50, len(task.target_dictionary))
            dec_args.lm_weight = cfg.wer_lm_weight
            dec_args.word_score = cfg.wer_word_score
            dec_args.unk_weight = -math.inf
            dec_args.sil_weight = 0

            self.w2l_decoder = W2lKenLMDecoder(dec_args, task.target_dictionary)
        else:
            self.w2l_decoder = None

        self.zero_infinity = cfg.zero_infinity
        self.sentence_avg = cfg.sentence_avg

        ## 구색 맞추기
        self.num_updates = 0

    def set_num_updates(self, num_updates):
        """Set the number of parameters updates."""
        self.num_updates = num_updates

    def forward(self, model, sample, reduce=True): #예측 결과와 실제 정답 비교, 평가지표로 계산
        net_output = model(**sample["net_input"])
        lprobs = model.get_normalized_probs( #예측값, softmax 된 확률값들
            net_output, log_probs=True
        ).contiguous()  # (T, B, C) from the encoder

        if "src_lengths" in sample["net_input"]: #패딩 부분 제외한 부분만 가져옴(앞뒤 패딩길이 무시)
            input_lengths = sample["net_input"]["src_lengths"]
        else:
            non_padding_mask = ~net_output["padding_mask"]
            input_lengths = non_padding_mask.long().sum(-1)

        pad_mask = (sample["target"] != self.pad_idx) & (
            sample["target"] != self.eos_idx
        )
        targets_flat = sample["target"].masked_select(pad_mask)

        if "target_lengths" in sample:
            target_lengths = sample["target_lengths"]
        else:
            target_lengths = pad_mask.sum(-1)


        # wav2vec2는 ctc loss만을 사용해서 학습합
        with torch.backends.cudnn.flags(enabled=False):
            loss = F.ctc_loss( #ctc loss 계산
                lprobs,
                targets_flat,
                input_lengths,
                target_lengths,
                blank=self.blank_idx,
                reduction="sum",
                zero_infinity=self.zero_infinity,
            )




        ntokens = (
            sample["ntokens"] if "ntokens" in sample else target_lengths.sum().item()
        )

        sample_size = sample["target"].size(0) if self.sentence_avg else ntokens
        logging_output = {
            "loss": utils.item(loss.data),  # * sample['ntokens'],
            "ntokens": ntokens,
            "nsentences": sample["id"].numel(),
            "sample_size": sample_size,
        }

        if not model.training: #eval 또는 validation인 경우
            import editdistance

            with torch.no_grad():
                lprobs_t = lprobs.transpose(0, 1).float().contiguous().cpu() #디코딩 준비

                c_err = 0
                c_len = 0
                w_errs = 0
                w_len = 0
                wv_errs = 0
                for lp, t, inp_l in zip( #각 샘플별 예측, 정답 비교(유효 범위, ctc 디코딩, 문자, 단어단위 오류 계산)
                    # lp = 하나의 샘플에 대한 모델 출력
                    # t = 해당 샘플의 정답 시퀸스
                    # inp_l = 입력 길이(실제 유효한 프레임의 길이)
                    lprobs_t,
                    sample["target_label"]
                    if "target_label" in sample
                    else sample["target"],
                    input_lengths,
                ):
                    lp = lp[:inp_l].unsqueeze(0) #inp_l 길이까지만 사용

                    decoded = None
                    if self.w2l_decoder is not None: #디코더 지정
                        decoded = self.w2l_decoder.decode(lp)
                        if len(decoded) < 1:
                            decoded = None
                        else:
                            decoded = decoded[0]
                            if len(decoded) < 1:
                                decoded = None
                            else:
                                decoded = decoded[0]

                    p = (t != self.task.target_dictionary.pad()) & ( #정답 전처리, pad랑 eos 삭제
                        t != self.task.target_dictionary.eos()
                    )
                    targ = t[p]
                    targ_units = self.task.target_dictionary.string(targ)
                    targ_units_arr = targ.tolist()

                    toks = lp.argmax(dim=-1).unique_consecutive() # ctc 중복 제거
                    pred_units_arr = toks[toks != self.blank_idx].tolist()#현재 샘플에서 가장 높은 확률의 토큰 선택

                    c_err += editdistance.eval(pred_units_arr, targ_units_arr)# 문자 단위 오류율 로스 계산
                    c_len += len(targ_units_arr)

                    targ_words = post_process(targ_units, self.post_process).split()# 단어 추가

                    pred_units = self.task.target_dictionary.string(pred_units_arr) #예측 token을 문자열로 변환
                    pred_words_raw = post_process(pred_units, self.post_process).split()

                    if decoded is not None and "words" in decoded:
                        #pred_words = beam search 결과
                        #prerd_words_raw = 단순 argmax 디코딩 결과
                        pred_words = decoded["words"]
                        w_errs += editdistance.eval(pred_words, targ_words)
                        wv_errs += editdistance.eval(pred_words_raw, targ_words)
                    else:
                        dist = editdistance.eval(pred_words_raw, targ_words)
                        w_errs += dist
                        wv_errs += dist

                    w_len += len(targ_words)

                logging_output["wv_errors"] = wv_errs
                logging_output["w_errors"] = w_errs
                logging_output["w_total"] = w_len
                logging_output["c_errors"] = c_err
                logging_output["c_total"] = c_len

        return loss, sample_size, logging_output

    @staticmethod
    def reduce_metrics(logging_outputs) -> None:
        """Aggregate logging outputs from data parallel training."""

        loss_sum = utils.item(sum(log.get("loss", 0) for log in logging_outputs))
        ntokens = utils.item(sum(log.get("ntokens", 0) for log in logging_outputs))
        nsentences = utils.item(
            sum(log.get("nsentences", 0) for log in logging_outputs)
        )
        sample_size = utils.item(
            sum(log.get("sample_size", 0) for log in logging_outputs)
        )

        metrics.log_scalar(
            "loss", loss_sum / sample_size / math.log(2), sample_size, round=3
        )
        metrics.log_scalar("ntokens", ntokens)
        metrics.log_scalar("nsentences", nsentences)
        if sample_size != ntokens:
            metrics.log_scalar(
                "nll_loss", loss_sum / ntokens / math.log(2), ntokens, round=3
            )

        c_errors = sum(log.get("c_errors", 0) for log in logging_outputs)
        metrics.log_scalar("_c_errors", c_errors)
        c_total = sum(log.get("c_total", 0) for log in logging_outputs)
        metrics.log_scalar("_c_total", c_total)
        w_errors = sum(log.get("w_errors", 0) for log in logging_outputs)
        metrics.log_scalar("_w_errors", w_errors)
        wv_errors = sum(log.get("wv_errors", 0) for log in logging_outputs)
        metrics.log_scalar("_wv_errors", wv_errors)
        w_total = sum(log.get("w_total", 0) for log in logging_outputs)
        metrics.log_scalar("_w_total", w_total)

        if c_total > 0:
            metrics.log_derived(
                "uer",
                lambda meters: safe_round(
                    meters["_c_errors"].sum * 100.0 / meters["_c_total"].sum, 3
                )
                if meters["_c_total"].sum > 0
                else float("nan"),
            )
        if w_total > 0:
            metrics.log_derived(
                "wer",
                lambda meters: safe_round(
                    meters["_w_errors"].sum * 100.0 / meters["_w_total"].sum, 3
                )
                if meters["_w_total"].sum > 0
                else float("nan"),
            )
            metrics.log_derived(
                "raw_wer",
                lambda meters: safe_round(
                    meters["_wv_errors"].sum * 100.0 / meters["_w_total"].sum, 3
                )
                if meters["_w_total"].sum > 0
                else float("nan"),
            )

    @staticmethod
    def logging_outputs_can_be_summed() -> bool:
        """
        Whether the logging outputs returned by `forward` can be summed
        across workers prior to calling `reduce_metrics`. Setting this
        to True will improves distributed training speed.
        """
        return True
#@register_criterion("ctc", dataclass=CtcCriterionConfig)
class CtcCriterion(FairseqCriterion):
    def __init__(self, cfg: CtcCriterionConfig, task: FairseqTask):
        super().__init__(task)
        self.blank_idx = task.target_dictionary.index(task.blank_symbol) if hasattr(task, 'blank_symbol') else 0
        self.pad_idx = task.target_dictionary.pad() #각각의 추가 토큰 인덱스 지정
        self.eos_idx = task.target_dictionary.eos()
        self.post_process = cfg.post_process #후처리

        if cfg.wer_args is not None:
            (
                cfg.wer_kenlm_model,
                cfg.wer_lexicon,
                cfg.wer_lm_weight,
                cfg.wer_word_score,
            ) = eval(cfg.wer_args)

        if cfg.wer_kenlm_model is not None:
            from examples.speech_recognition.w2l_decoder import W2lKenLMDecoder

            dec_args = Namespace()
            dec_args.nbest = 1
            dec_args.criterion = "ctc"
            dec_args.kenlm_model = cfg.wer_kenlm_model
            dec_args.lexicon = cfg.wer_lexicon
            dec_args.beam = 50
            dec_args.beam_size_token = min(50, len(task.target_dictionary))
            dec_args.beam_threshold = min(50, len(task.target_dictionary))
            dec_args.lm_weight = cfg.wer_lm_weight
            dec_args.word_score = cfg.wer_word_score
            dec_args.unk_weight = -math.inf
            dec_args.sil_weight = 0

            self.w2l_decoder = W2lKenLMDecoder(dec_args, task.target_dictionary)
        else:
            self.w2l_decoder = None

        self.zero_infinity = cfg.zero_infinity
        self.sentence_avg = cfg.sentence_avg

        ## 구색 맞추기
        self.num_updates = 0

    def set_num_updates(self, num_updates):
        """Set the number of parameters updates."""
        self.num_updates = num_updates

    def forward(self, model, sample, reduce=True):
        net_output = model(**sample["net_input"])
        lprobs = model.get_normalized_probs(
            net_output, log_probs=True
        ).contiguous()  # (T, B, C) from the encoder

        if "src_lengths" in sample["net_input"]:
            input_lengths = sample["net_input"]["src_lengths"]
        else:
            non_padding_mask = ~net_output["padding_mask"]
            input_lengths = non_padding_mask.long().sum(-1)

        pad_mask = (sample["target"] != self.pad_idx) & (
            sample["target"] != self.eos_idx
        )
        targets_flat = sample["target"].masked_select(pad_mask)

        if "target_lengths" in sample:
            target_lengths = sample["target_lengths"]
        else:
            target_lengths = pad_mask.sum(-1)

        with torch.backends.cudnn.flags(enabled=False):
            loss = F.ctc_loss(
                lprobs,
                targets_flat,
                input_lengths,
                target_lengths,
                blank=self.blank_idx,
                reduction="sum",
                zero_infinity=self.zero_infinity,
            )

        ntokens = (
            sample["ntokens"] if "ntokens" in sample else target_lengths.sum().item()
        )

        sample_size = sample["target"].size(0) if self.sentence_avg else ntokens
        logging_output = {
            "loss": utils.item(loss.data),  # * sample['ntokens'],
            "ntokens": ntokens,
            "nsentences": sample["id"].numel(),
            "sample_size": sample_size,
        }

        if not model.training:
            import editdistance

            with torch.no_grad():
                lprobs_t = lprobs.transpose(0, 1).float().contiguous().cpu()

                c_err = 0
                c_len = 0
                w_errs = 0
                w_len = 0
                wv_errs = 0
                for lp, t, inp_l in zip(
                    lprobs_t,
                    sample["target_label"]
                    if "target_label" in sample
                    else sample["target"],
                    input_lengths,
                ):
                    lp = lp[:inp_l].unsqueeze(0)

                    decoded = None
                    if self.w2l_decoder is not None:
                        decoded = self.w2l_decoder.decode(lp)
                        if len(decoded) < 1:
                            decoded = None
                        else:
                            decoded = decoded[0]
                            if len(decoded) < 1:
                                decoded = None
                            else:
                                decoded = decoded[0]

                    p = (t != self.task.target_dictionary.pad()) & (
                        t != self.task.target_dictionary.eos()
                    )
                    targ = t[p]
                    targ_units = self.task.target_dictionary.string(targ)
                    targ_units_arr = targ.tolist()

                    toks = lp.argmax(dim=-1).unique_consecutive()
                    pred_units_arr = toks[toks != self.blank_idx].tolist()

                    c_err += editdistance.eval(pred_units_arr, targ_units_arr)
                    c_len += len(targ_units_arr)

                    targ_words = post_process(targ_units, self.post_process).split()

                    pred_units = self.task.target_dictionary.string(pred_units_arr)
                    pred_words_raw = post_process(pred_units, self.post_process).split()

                    if decoded is not None and "words" in decoded:
                        pred_words = decoded["words"]
                        w_errs += editdistance.eval(pred_words, targ_words)
                        wv_errs += editdistance.eval(pred_words_raw, targ_words)
                    else:
                        dist = editdistance.eval(pred_words_raw, targ_words)
                        w_errs += dist
                        wv_errs += dist

                    w_len += len(targ_words)

                logging_output["wv_errors"] = wv_errs
                logging_output["w_errors"] = w_errs
                logging_output["w_total"] = w_len
                logging_output["c_errors"] = c_err
                logging_output["c_total"] = c_len

        return loss, sample_size, logging_output

    @staticmethod
    def reduce_metrics(logging_outputs) -> None:
        """Aggregate logging outputs from data parallel training."""

        loss_sum = utils.item(sum(log.get("loss", 0) for log in logging_outputs))
        ntokens = utils.item(sum(log.get("ntokens", 0) for log in logging_outputs))
        nsentences = utils.item(
            sum(log.get("nsentences", 0) for log in logging_outputs)
        )
        sample_size = utils.item(
            sum(log.get("sample_size", 0) for log in logging_outputs)
        )

        metrics.log_scalar(
            "loss", loss_sum / sample_size / math.log(2), sample_size, round=3
        )
        metrics.log_scalar("ntokens", ntokens)
        metrics.log_scalar("nsentences", nsentences)
        if sample_size != ntokens:
            metrics.log_scalar(
                "nll_loss", loss_sum / ntokens / math.log(2), ntokens, round=3
            )

        c_errors = sum(log.get("c_errors", 0) for log in logging_outputs)
        metrics.log_scalar("_c_errors", c_errors)
        c_total = sum(log.get("c_total", 0) for log in logging_outputs)
        metrics.log_scalar("_c_total", c_total)
        w_errors = sum(log.get("w_errors", 0) for log in logging_outputs)
        metrics.log_scalar("_w_errors", w_errors)
        wv_errors = sum(log.get("wv_errors", 0) for log in logging_outputs)
        metrics.log_scalar("_wv_errors", wv_errors)
        w_total = sum(log.get("w_total", 0) for log in logging_outputs)
        metrics.log_scalar("_w_total", w_total)

        if c_total > 0:
            metrics.log_derived(
                "uer",
                lambda meters: safe_round(
                    meters["_c_errors"].sum * 100.0 / meters["_c_total"].sum, 3
                )
                if meters["_c_total"].sum > 0
                else float("nan"),
            )
        if w_total > 0:
            metrics.log_derived(
                "wer",
                lambda meters: safe_round(
                    meters["_w_errors"].sum * 100.0 / meters["_w_total"].sum, 3
                )
                if meters["_w_total"].sum > 0
                else float("nan"),
            )
            metrics.log_derived(
                "raw_wer",
                lambda meters: safe_round(
                    meters["_wv_errors"].sum * 100.0 / meters["_w_total"].sum, 3
                )
                if meters["_w_total"].sum > 0
                else float("nan"),
            )

    @staticmethod
    def logging_outputs_can_be_summed() -> bool:
        """
        Whether the logging outputs returned by `forward` can be summed
        across workers prior to calling `reduce_metrics`. Setting this
        to True will improves distributed training speed.
        """
        return True

#trainer.py


In [21]:
logger = logging.getLogger(__name__)


class Trainer(object):
    """Main class for data parallel training.

    This class supports synchronous distributed data parallel training,
    where multiple workers each have a full model replica and gradients
    are accumulated across workers before each update. We use
    :class:`~torch.nn.parallel.DistributedDataParallel` to handle
    communication of the gradients across workers.
    """

    def __init__(self, cfg: FairseqConfig, task, model, criterion, quantizer=None):

        if isinstance(cfg, Namespace):
            logger.warning(
                "argparse.Namespace configuration is deprecated! Automatically converting to OmegaConf"
            )
            cfg = convert_namespace_to_omegaconf(cfg)

        self.cfg = cfg
        self.task = task

        # catalog shared parameters
        shared_params = _catalog_shared_params(model)
        self.tpu = cfg.common.tpu
        self.cuda = torch.cuda.is_available() and not cfg.common.cpu and not self.tpu
        if self.cuda:
            self.device = torch.device("cuda")
        elif self.tpu:
            self.device = utils.get_tpu_device()
        else:
            self.device = torch.device("cpu")

        # copy model and criterion to current device/dtype
        self._criterion = criterion #평가 함수 지정
        self._model = model
        if cfg.common.fp16: #precision 지정
            self._criterion = self._criterion.half()
            self._model = self._model.half()
        elif cfg.common.bf16:
            self._criterion = self._criterion.to(dtype=torch.bfloat16)
            self._model = self._model.to(dtype=torch.bfloat16)
        if not cfg.distributed_training.pipeline_model_parallel: #병렬모델 여부
            self._criterion = self._criterion.to(device=self.device)
            self._model = self._model.to(device=self.device)

        self.pipeline_model_parallel = cfg.distributed_training.pipeline_model_parallel
        self.last_device = None

        if self.cuda and self.pipeline_model_parallel:
            self.last_device = torch.device(
                cfg.distributed_training.pipeline_devices[-1]
            )

        # check that shared parameters are preserved after device transfer
        for shared_param in shared_params:
            ref = _get_module_by_path(self._model, shared_param[0])
            for path in shared_param[1:]:
                logger.info(
                    "detected shared parameter: {} <- {}".format(shared_param[0], path)
                )
                _set_module_by_path(self._model, path, ref)

        self._dummy_batch = None  # 모양 확인용 가짜 배치
        self._lr_scheduler = None # 학습률 조절기
        self._num_updates = 0 #옵티마이저 업데이트 횟수
        self._num_xla_compiles = 0  # for TPUs
        self._optim_history = None #옵티마이저 상태 기록
        self._optimizer = None #옵티마이저
        self._warn_once = set() #경고메세지 중복 방지용 세트
        self._wrapped_criterion = None # wrap된 loss function 객체
        self._wrapped_model = None #wrap된 모델 객체(추가 기능이 추가된 모델)

        # TODO(myleott): support tpu
        if self.cuda and self.data_parallel_world_size > 1:
            self._grad_norm_buf = torch.cuda.DoubleTensor(self.data_parallel_world_size)
        else:
            self._grad_norm_buf = None

        self.quantizer = quantizer
        if self.quantizer is not None:
            self.quantizer.set_trainer(self)

        # get detailed cuda environment
        if self.cuda:
            self.cuda_env = utils.CudaEnvironment()
            if self.data_parallel_world_size > 1:
                self.cuda_env_arr = distributed_utils.all_gather_list(
                    self.cuda_env, group=distributed_utils.get_global_group()
                )
            else:
                self.cuda_env_arr = [self.cuda_env]
            if self.data_parallel_rank == 0:
                utils.CudaEnvironment.pretty_print_cuda_env_list(self.cuda_env_arr)
        else:
            self.cuda_env = None
            self.cuda_env_arr = None

        metrics.log_start_time("wall", priority=790, round=0)

        self._start_time = time.time()
        self._previous_training_time = 0
        self._cumulative_training_time = None

    def reinitialize(self): #재초기화
        """Reinitialize the Trainer, typically after model params change."""
        self._lr_scheduler = None
        self._optimizer = None
        self._wrapped_criterion = None
        self._wrapped_model = None

    @property
    def data_parallel_world_size(self): #분산 병렬 학습용 식별 코드
        if self.cfg.distributed_training.distributed_world_size == 1:
            return 1
        return distributed_utils.get_data_parallel_world_size()

    @property
    def data_parallel_process_group(self):
        return distributed_utils.get_data_parallel_group()

    @property
    def data_parallel_rank(self):
        if self.cfg.distributed_training.distributed_world_size == 1:
            return 0
        return distributed_utils.get_data_parallel_rank()

    @property
    def is_data_parallel_master(self):
        # NOTE: this returns true for all model parallel replicas with data
        # parallel rank 0
        return self.data_parallel_rank == 0

    @property
    def criterion(self):
        if self._wrapped_criterion is None:
            if (
                utils.has_parameters(self._criterion)
                and self.data_parallel_world_size > 1
                and not self.cfg.optimization.use_bmuf
            ):
                self._wrapped_criterion = models.DistributedFairseqModel(
                    self.cfg.distributed_training,
                    self._criterion,
                    process_group=self.data_parallel_process_group,
                )
            else:
                self._wrapped_criterion = self._criterion
        return self._wrapped_criterion

    @property
    def model(self):
        if self._wrapped_model is None:
            if self.data_parallel_world_size > 1 and not self.cfg.optimization.use_bmuf:
                self._wrapped_model = models.DistributedFairseqModel(
                    self.cfg.distributed_training,
                    self._model,
                    process_group=self.data_parallel_process_group,
                )
            else:
                self._wrapped_model = self._model
        return self._wrapped_model

    @property
    def optimizer(self):
        if self._optimizer is None:
            self._build_optimizer()
        return self._optimizer

    @property
    def lr_scheduler(self):
        if self._lr_scheduler is None:
            self._build_optimizer()  # this will initialize self._lr_scheduler
        return self._lr_scheduler

    def _build_optimizer(self):
        params = list(
            filter(
                lambda p: p.requires_grad,
                chain(self.model.parameters(), self.criterion.parameters()),
            )
        )

        if self.cfg.common.fp16 or self.cfg.common.bf16:
            if self.cuda and torch.cuda.get_device_capability(0)[0] < 7:
                logger.info(
                    "NOTE: your device does NOT support faster training with --fp16, "
                    "please switch to FP32 which is likely to be faster"
                )
            if (
                self.cfg.common.memory_efficient_fp16
                or self.cfg.common.memory_efficient_bf16
            ):
                self._optimizer = optim.MemoryEfficientFP16Optimizer.build_optimizer(
                    self.cfg, params
                )
            else:
                self._optimizer = optim.FP16Optimizer.build_optimizer(self.cfg, params)
        else:
            if self.cuda and torch.cuda.get_device_capability(0)[0] >= 7:
                logger.info("NOTE: your device may support faster training with --fp16")
            self._optimizer = optim.build_optimizer(self.cfg.optimizer, params)

        if self.cfg.optimization.use_bmuf:
            self._optimizer = optim.FairseqBMUF(
                self.cfg.bmuf,
                self._optimizer,
            )

        if self.cfg.distributed_training.zero_sharding == "os":
            if (
                self.cfg.common.fp16
                and not self.cfg.common.memory_efficient_fp16
                and not self.cfg.common.memory_efficient_bf16
            ) and not self.cfg.common.fp16_no_flatten_grads:
                raise ValueError(
                    "ZeRO is incomptabile with fp16 and flattened grads. "
                    "Please use --fp16-no-flatten-grads"
                )
            else:
                optim.shard_(self._optimizer, self.data_parallel_process_group)

        # We should initialize the learning rate scheduler immediately after
        # building the optimizer, so that the initial learning rate is set.
        self._lr_scheduler = lr_scheduler.build_lr_scheduler(
            self.cfg.lr_scheduler,
            self.optimizer,
        )
        self._lr_scheduler.step_update(0)

    def consolidate_optimizer(self):
        """For OSS, we need to consolidate the state dict."""
        if hasattr(self.optimizer.optimizer, "consolidate_state_dict"):
            self.optimizer.optimizer.consolidate_state_dict()

    def save_checkpoint(self, filename, extra_state):
        """Save all training state in a checkpoint file."""
        if self.is_data_parallel_master:  # only save one checkpoint
            extra_state["metrics"] = metrics.state_dict()
            extra_state["previous_training_time"] = self.cumulative_training_time()
            checkpoint_utils.save_state(
                filename,
                self.cfg,
                self.get_model().state_dict(),
                self.get_criterion(),
                self.optimizer,
                self.lr_scheduler,
                self.get_num_updates(),
                self._optim_history,
                extra_state,
            )
            logger.info(f"Finished saving checkpoint to {filename}")

    def load_checkpoint(
        self,
        filename,
        reset_optimizer=False,
        reset_lr_scheduler=False,
        optimizer_overrides=None,
        reset_meters=False,
    ):
        """
        Load all training state from a checkpoint file.
        rank = 0 will load the checkpoint, and then broadcast it to all
        other ranks.
        """
        extra_state, self._optim_history, last_optim_state = None, [], None

        logger.info(f"Preparing to load checkpoint {filename}")
        is_distributed = self.data_parallel_world_size > 1
        bexists = PathManager.isfile(filename)
        if bexists:
            load_on_all_ranks = (
                self.cfg.checkpoint.load_checkpoint_on_all_dp_ranks
                # TPUs don't support broadcast yet, so load checkpoints
                # on every worker for now
                or self.tpu
            )

            if load_on_all_ranks or self.data_parallel_rank == 0:
                state = checkpoint_utils.load_checkpoint_to_cpu(
                    filename, load_on_all_ranks=load_on_all_ranks
                )
                last_optim_state = state.get("last_optimizer_state", None)

                # If doing zero_sharding, do not broadcast global optimizer
                # state. Later we will broadcast sharded states to each rank
                # to avoid memory from exploding.
                if (
                    not load_on_all_ranks
                    and self.cfg.distributed_training.zero_sharding == "os"
                    and "last_optimizer_state" in state
                    and is_distributed
                ):
                    state["last_optimizer_state"] = "SHARDED"
            else:
                last_optim_state = None
                state = None

            if is_distributed and not load_on_all_ranks:
                state = distributed_utils.broadcast_object(
                    state,
                    src_rank=0,
                    group=self.data_parallel_process_group,
                    dist_device=self.device,
                )
                if self.data_parallel_rank > 0:
                    last_optim_state = state.get("last_optimizer_state", None)

            # load model parameters
            try:
                self.get_model().load_state_dict(
                    state["model"], strict=True, model_cfg=self.cfg.model
                )
                if utils.has_parameters(self.get_criterion()):
                    self.get_criterion().load_state_dict(
                        state["criterion"], strict=True
                    )
            except Exception:
                raise Exception(
                    "Cannot load model parameters from checkpoint {}; "
                    "please ensure that the architectures match.".format(filename)
                )
            extra_state = state["extra_state"]
            self._optim_history = state["optimizer_history"]

        if last_optim_state is not None and not reset_optimizer:
            # rebuild optimizer after loading model, since params may have changed
            self._build_optimizer()

            # only reload optimizer and lr_scheduler if they match
            last_optim = self._optim_history[-1]
            assert (
                last_optim["criterion_name"] == self.get_criterion().__class__.__name__
            ), "Criterion does not match; please reset the optimizer (--reset-optimizer)."
            assert (
                last_optim["optimizer_name"] == self.optimizer.__class__.__name__
            ), "Optimizer does not match; please reset the optimizer (--reset-optimizer)."

            if not reset_lr_scheduler:
                self.lr_scheduler.load_state_dict(last_optim["lr_scheduler_state"])

            if not load_on_all_ranks and is_distributed:
                last_optim_state = self.optimizer.broadcast_global_state_dict(
                    last_optim_state
                )
            self.optimizer.load_state_dict(last_optim_state, optimizer_overrides)

            self.set_num_updates(last_optim["num_updates"])

        if extra_state is not None:
            itr_state = extra_state["train_iterator"]
            epoch = itr_state["epoch"]

            if "previous_training_time" in extra_state:
                self._previous_training_time = extra_state["previous_training_time"]
                self._start_time = time.time()

            self.lr_step(epoch)

            if itr_state.get("version", 1) >= 2 and itr_state["iterations_in_epoch"] == 0:
                # reset meters at start of epoch
                reset_meters = True

            if "metrics" in extra_state and not reset_meters:
                metrics.load_state_dict(extra_state["metrics"])

                # reset TimeMeters, since their start times don't make sense anymore
                for meter in metrics.get_meters("default"):
                    if isinstance(meter, meters.TimeMeter):
                        meter.reset()

            logger.info(
                "Loaded checkpoint {} (epoch {} @ {} updates)".format(
                    filename, epoch, self.get_num_updates()
                )
            )

        else:
            logger.info("No existing checkpoint found {}".format(filename))

        return extra_state

    def get_train_iterator(
        self,
        epoch,
        combine=True,
        load_dataset=True,
        data_selector=None,
        shard_batch_itr=True,
        disable_iterator_cache=False,
    ):
        """Return an EpochBatchIterator over the training set for a given epoch."""
        if load_dataset:
            logger.info("loading train data for epoch {}".format(epoch))
            self.task.load_dataset( #데이터셋 로딩
                self.cfg.dataset.train_subset,
                epoch=epoch,
                combine=combine,
                data_selector=data_selector,
            )
        batch_iterator = self.task.get_batch_iterator( #배치 생성기 초기화
            dataset=self.task.dataset(self.cfg.dataset.train_subset),
            max_tokens=self.cfg.dataset.max_tokens,
            max_sentences=self.cfg.dataset.batch_size,
            max_positions=utils.resolve_max_positions(
                self.task.max_positions(),
                self.model.max_positions(),
                self.cfg.dataset.max_tokens,
            ),
            ignore_invalid_inputs=True,
            required_batch_size_multiple=self.cfg.dataset.required_batch_size_multiple,
            seed=self.cfg.common.seed,
            num_shards=self.data_parallel_world_size if shard_batch_itr else 1,
            shard_id=self.data_parallel_rank if shard_batch_itr else 0,
            num_workers=self.cfg.dataset.num_workers,
            epoch=epoch,
            data_buffer_size=self.cfg.dataset.data_buffer_size,
            disable_iterator_cache=disable_iterator_cache,
        )
        self.reset_dummy_batch(batch_iterator.first_batch)
        return batch_iterator

    def get_valid_iterator( #검증 생성
        self,
        subset,
        disable_iterator_cache=False,
    ):
        """Return an EpochBatchIterator over given validation subset for a given epoch."""
        batch_iterator = self.task.get_batch_iterator(
            dataset=self.task.dataset(subset),
            max_tokens=self.cfg.dataset.max_tokens_valid,
            max_sentences=self.cfg.dataset.batch_size_valid,
            max_positions=utils.resolve_max_positions(
                self.task.max_positions(),
                self.model.max_positions(),
            ),
            ignore_invalid_inputs=self.cfg.dataset.skip_invalid_size_inputs_valid_test,
            required_batch_size_multiple=self.cfg.dataset.required_batch_size_multiple,
            seed=self.cfg.common.seed,
            num_shards=self.data_parallel_world_size,
            shard_id=self.data_parallel_rank,
            num_workers=self.cfg.dataset.num_workers,
            # always pass a fixed "epoch" to keep validation data consistent
            # across training epochs
            epoch=1,
            data_buffer_size=self.cfg.dataset.data_buffer_size,
            disable_iterator_cache=disable_iterator_cache,
        )
        self.reset_dummy_batch(batch_iterator.first_batch)
        return batch_iterator

    def begin_epoch(self, epoch): #에폭 시작
        """Called at the beginning of each epoch."""
        logger.info("begin training epoch {}".format(epoch))

        self.lr_step_begin_epoch(epoch)

        if self.quantizer is not None:
            self.quantizer.begin_epoch(epoch)

        # task specific setup per epoch
        self.task.begin_epoch(epoch, self.get_model())

        if self.tpu:
            import torch_xla.core.xla_model as xm

            xm.rendezvous("begin_epoch")  # wait for all workers
            xm.mark_step()

    def begin_valid_epoch(self, epoch):
        """Called at the beginning of each validation epoch."""

        # task specific setup per validation epoch
        self.task.begin_valid_epoch(epoch, self.get_model())

    def reset_dummy_batch(self, batch):
        self._dummy_batch = batch

    @metrics.aggregate("train")
    def train_step(self, samples, raise_oom=False): #학습 스텝 지정
        """Do forward, backward and parameter update."""
        self._set_seed()
        self.model.train() #학습
        self.criterion.train()
        self.zero_grad()

        metrics.log_start_time("train_wall", priority=800, round=0)

        # forward and backward pass
        logging_outputs, sample_size, ooms = [], 0, 0
        for i, sample in enumerate(samples):  # delayed update loop
            sample, is_dummy_batch = self._prepare_sample(sample)

            def maybe_no_sync():
                """
                Whenever *samples* contains more than one mini-batch, we
                want to accumulate gradients locally and only call
                all-reduce in the last backwards pass.
                """
                if (
                    self.data_parallel_world_size > 1
                    and hasattr(self.model, "no_sync")
                    and i < len(samples) - 1
                ):
                    return self.model.no_sync()
                else:
                    return contextlib.ExitStack()  # dummy contextmanager

            try:
                with maybe_no_sync():
                    # forward and backward
                    # 학습 진행하면서 loss 값 사용
                    loss, sample_size_i, logging_output = self.task.train_step(
                        sample=sample,
                        model=self.model,
                        criterion=self.criterion,
                        optimizer=self.optimizer,
                        update_num=self.get_num_updates(),
                        ignore_grad=is_dummy_batch,
                    )
                    del loss

                logging_outputs.append(logging_output)
                sample_size += sample_size_i

                # emptying the CUDA cache after the first step can
                # reduce the chance of OOM
                if self.cuda and self.get_num_updates() == 0:
                    torch.cuda.empty_cache()
            except RuntimeError as e:
                if "out of memory" in str(e):
                    self._log_oom(e)
                    if raise_oom:
                        raise e
                    logger.warning(
                        "attempting to recover from OOM in forward/backward pass"
                    )
                    ooms += 1
                    self.zero_grad()
                    if self.cuda:
                        torch.cuda.empty_cache()
                    if self.cfg.distributed_training.distributed_world_size == 1:
                        return None
                else:
                    raise e

            if self.tpu and i < len(samples) - 1:
                # tpu-comment: every XLA operation before marking step is
                # appended to the IR graph, and processing too many batches
                # before marking step can lead to OOM errors.
                # To handle gradient accumulation use case, we explicitly
                # mark step here for every forward pass without a backward pass
                import torch_xla.core.xla_model as xm

                xm.mark_step()

        if is_dummy_batch: #더미 배치
            if torch.is_tensor(sample_size):
                sample_size.zero_()
            else:
                sample_size *= 0.0

        if torch.is_tensor(sample_size):
            sample_size = sample_size.float()
        else:
            sample_size = float(sample_size)

        # gather logging outputs from all replicas
        if self._sync_stats():
            train_time = self._local_cumulative_training_time()
            logging_outputs, (
                sample_size,
                ooms,
                total_train_time,
            ) = self._aggregate_logging_outputs(
                logging_outputs,
                sample_size,
                ooms,
                train_time,
                ignore=is_dummy_batch,
            )
            self._cumulative_training_time = (
                total_train_time / self.data_parallel_world_size
            )

        overflow = False
        try:
            with torch.autograd.profiler.record_function("reduce-grads"):
                # reduce gradients across workers
                self.optimizer.all_reduce_grads(self.model)
                if utils.has_parameters(self.criterion):
                    self.optimizer.all_reduce_grads(self.criterion)

            with torch.autograd.profiler.record_function("multiply-grads"):
                # multiply gradients by (data_parallel_size / sample_size) since
                # DDP normalizes by the number of data parallel workers for
                # improved fp16 precision.
                # Thus we get (sum_of_gradients / sample_size) at the end.
                # In case of fp16, this step also undoes loss scaling.
                # (Debugging note: Some optimizers perform this scaling on the
                # fly, so inspecting model.parameters() or optimizer.params may
                # still show the original, unscaled gradients.)
                numer = (
                    self.data_parallel_world_size
                    if not self.cfg.optimization.use_bmuf or self._sync_stats()
                    else 1
                )
                self.optimizer.multiply_grads(numer / (sample_size or 1.0))
                # Note: (sample_size or 1.0) handles the case of a zero gradient, in a
                # way that avoids CPU/device transfers in case sample_size is a GPU or
                # TPU object. The assumption is that the gradient itself is also 0.

            with torch.autograd.profiler.record_function("clip-grads"):
                # clip grads
                grad_norm = self.clip_grad_norm(self.cfg.optimization.clip_norm)

            # check that grad norms are consistent across workers
            # on tpu check tensor is slow
            if not self.tpu:
                if (
                    not self.cfg.optimization.use_bmuf
                    and self.cfg.distributed_training.distributed_wrapper != "SlowMo"
                ):
                    self._check_grad_norms(grad_norm)
                if not torch.isfinite(grad_norm).all():
                    # check local gradnorm single GPU case, trigger NanDetector
                    raise FloatingPointError("gradients are Nan/Inf")

            with torch.autograd.profiler.record_function("optimizer"):
                # take an optimization step
                self.task.optimizer_step(
                    self.optimizer, model=self.model, update_num=self.get_num_updates()
                )

        except FloatingPointError:
            # re-run the forward and backward pass with hooks attached to print
            # out where it fails
            self.zero_grad()
            with NanDetector(self.get_model()):
                for _, sample in enumerate(samples):
                    sample, _ = self._prepare_sample(sample)
                    self.task.train_step(
                        sample,
                        self.model,
                        self.criterion,
                        self.optimizer,
                        self.get_num_updates(),
                        ignore_grad=False,
                    )
            raise
        except OverflowError as e:
            overflow = True
            logger.info(f"NOTE: gradient overflow detected, ignoring gradient, {str(e)}")
            grad_norm = torch.tensor(0.0).cuda()
            self.zero_grad()
        except RuntimeError as e:
            if "out of memory" in str(e):
                self._log_oom(e)
                logger.error("OOM during optimization, irrecoverable")
            raise e

        # Some distributed wrappers (e.g., SlowMo) need access to the optimizer after the step
        if hasattr(self.model, "perform_additional_optimizer_actions"):
            if hasattr(self.optimizer, "fp32_params"):
                self.model.perform_additional_optimizer_actions(
                    self.optimizer.optimizer, self.optimizer.fp32_params
                )
            else:
                self.model.perform_additional_optimizer_actions(
                    self.optimizer.optimizer
                )

        logging_output = None
        if (
            not overflow
            or self.cfg.distributed_training.distributed_wrapper == "SlowMo"
        ):
            self.set_num_updates(self.get_num_updates() + 1)

            if self.tpu:
                # mark step on TPUs
                import torch_xla.core.xla_model as xm

                xm.mark_step()

                # only log stats every log_interval steps
                # this causes wps to be misreported when log_interval > 1
                logging_output = {}
                if self.get_num_updates() % self.cfg.common.log_interval == 0:
                    # log memory usage
                    mem_info = xm.get_memory_info(self.device)
                    gb_free = mem_info["kb_free"] / 1024 / 1024
                    gb_total = mem_info["kb_total"] / 1024 / 1024
                    metrics.log_scalar(
                        "gb_free",
                        gb_free,
                        priority=1500,
                        round=1,
                        weight=0,
                    )
                    metrics.log_scalar(
                        "gb_total",
                        gb_total,
                        priority=1600,
                        round=1,
                        weight=0,
                    )

                    logging_output = self._reduce_and_log_stats(
                        logging_outputs,
                        sample_size,
                        grad_norm,
                    )

                # log whenever there's an XLA compilation, since these
                # slow down training and may indicate opportunities for
                # optimization
                self._check_xla_compilation()
            else:
                # log stats
                logging_output = self._reduce_and_log_stats(
                    logging_outputs,
                    sample_size,
                    grad_norm,
                )

                # clear CUDA cache to reduce memory fragmentation
                if (
                    self.cuda
                    and self.cfg.common.empty_cache_freq > 0
                    and (
                        (self.get_num_updates() + self.cfg.common.empty_cache_freq - 1)
                        % self.cfg.common.empty_cache_freq
                    )
                    == 0
                ):
                    torch.cuda.empty_cache()

        if self.cfg.common.fp16:
            metrics.log_scalar(
                "loss_scale",
                self.optimizer.scaler.loss_scale,
                priority=700,
                round=4,
                weight=0,
            )

        metrics.log_stop_time("train_wall")
        return logging_output

    @metrics.aggregate("valid")
    def valid_step(self, sample, raise_oom=False): #valid step
        """Do forward pass in evaluation mode."""
        if self.tpu:
            import torch_xla.core.xla_model as xm

            xm.rendezvous("valid_step")  # wait for all workers
            xm.mark_step()

        with torch.no_grad():
            self.model.eval()
            self.criterion.eval()

            sample, is_dummy_batch = self._prepare_sample(sample)

            try:
                _loss, sample_size, logging_output = self.task.valid_step(
                    sample, self.model, self.criterion
                )
            except RuntimeError as e:
                if "out of memory" in str(e):
                    self._log_oom(e)
                    if not raise_oom:
                        logger.warning(
                            "ran out of memory in validation step, retrying batch"
                        )
                        for p in self.model.parameters():
                            if p.grad is not None:
                                p.grad = None  # free some memory
                        if self.cuda:
                            torch.cuda.empty_cache()
                        return self.valid_step(sample, raise_oom=True)
                raise e

            logging_outputs = [logging_output]
            if is_dummy_batch:
                if torch.is_tensor(sample_size):
                    sample_size.zero_()
                else:
                    sample_size *= 0.0

        # gather logging outputs from all replicas
        if self.data_parallel_world_size > 1:
            logging_outputs, (sample_size,) = self._aggregate_logging_outputs(
                logging_outputs,
                sample_size,
                ignore=is_dummy_batch,
            )

        # log validation stats
        logging_output = self._reduce_and_log_stats(logging_outputs, sample_size)

        return logging_output

    def zero_grad(self):
        self.optimizer.zero_grad()

    def lr_step_begin_epoch(self, epoch):
        """Adjust the learning rate at the beginning of the epoch."""
        self.lr_scheduler.step_begin_epoch(epoch)
        # prefer updating the LR based on the number of steps
        return self.lr_step_update()

    def lr_step(self, epoch, val_loss=None):
        """Adjust the learning rate at the end of the epoch."""
        self.lr_scheduler.step(epoch, val_loss)
        # prefer updating the LR based on the number of steps
        return self.lr_step_update()

    def lr_step_update(self):
        """Update the learning rate after each update."""
        new_lr = self.lr_scheduler.step_update(self.get_num_updates())
        if isinstance(new_lr, dict):
            for k, v in new_lr.items():
                metrics.log_scalar(f"lr_{k}", v, weight=0, priority=300)
            new_lr = new_lr.get("default", next(iter(new_lr.values())))
        else:
            metrics.log_scalar("lr", new_lr, weight=0, priority=300)
        return new_lr

    def get_lr(self):
        """Get the current learning rate."""
        return self.optimizer.get_lr()

    def get_model(self):
        """Get the (non-wrapped) model instance."""
        return self._model

    def get_criterion(self):
        """Get the (non-wrapped) criterion instance."""
        return self._criterion

    def get_meter(self, name):
        """[deprecated] Get a specific meter by name."""
        from fairseq import meters

        if "get_meter" not in self._warn_once:
            self._warn_once.add("get_meter")
            utils.deprecation_warning(
                "Trainer.get_meter is deprecated. Please use fairseq.metrics instead."
            )

        train_meters = metrics.get_meters("train")
        if train_meters is None:
            train_meters = {}

        if name == "train_loss" and "loss" in train_meters:
            return train_meters["loss"]
        elif name == "train_nll_loss":
            # support for legacy train.py, which assumed this meter is
            # always initialized
            m = train_meters.get("nll_loss", None)
            return m or meters.AverageMeter()
        elif name == "wall":
            # support for legacy train.py, which assumed this meter is
            # always initialized
            m = metrics.get_meter("default", "wall")
            return m or meters.TimeMeter()
        elif name == "wps":
            m = metrics.get_meter("train", "wps")
            return m or meters.TimeMeter()
        elif name in {"valid_loss", "valid_nll_loss"}:
            # support for legacy train.py, which assumed these meters
            # are always initialized
            k = name[len("valid_") :]
            m = metrics.get_meter("valid", k)
            return m or meters.AverageMeter()
        elif name == "oom":
            return meters.AverageMeter()
        elif name in train_meters:
            return train_meters[name]
        return None

    def get_num_updates(self):
        """Get the number of parameters updates."""
        return self._num_updates

    def set_num_updates(self, num_updates):
        """Set the number of parameters updates."""
        self._num_updates = num_updates
        self.lr_step_update()
        if self.quantizer:
            self.quantizer.step_update(self._num_updates)
        metrics.log_scalar("num_updates", self._num_updates, weight=0, priority=200)

    def clip_grad_norm(self, clip_norm):
        return self.optimizer.clip_grad_norm(clip_norm, aggregate_norm_fn=None)

    def cumulative_training_time(self):
        if self._cumulative_training_time is None:
            # single GPU
            return self._local_cumulative_training_time()
        else:
            return self._cumulative_training_time

    def _local_cumulative_training_time(self):
        """Aggregate training time in seconds."""
        return time.time() - self._start_time + self._previous_training_time

    def _prepare_sample(self, sample, is_dummy=False):
        if sample == "DUMMY":
            raise Exception(
                "Trying to use an uninitialized 'dummy' batch. This usually indicates "
                "that the total number of batches is smaller than the number of "
                "participating GPUs. Try reducing the batch size or using fewer GPUs."
            )

        if sample is None or len(sample) == 0:
            assert (
                self._dummy_batch is not None and len(self._dummy_batch) > 0
            ), "Invalid dummy batch: {}".format(self._dummy_batch)
            sample, _ = self._prepare_sample(self._dummy_batch, is_dummy=True)
            return sample, True

        if self.cuda:
            if self.pipeline_model_parallel:
                if "target" in sample:
                    sample["target"] = utils.move_to_cuda(
                        sample["target"], device=self.last_device
                    )
            else:
                sample = utils.move_to_cuda(sample)
        elif self.tpu and is_dummy:
            # the dummy batch may not be on the appropriate device
            sample = utils.move_to_cuda(sample, device=self.device)

        def apply_half(t):
            if t.dtype is torch.float32:
                return t.half()
            return t

        def apply_bfloat16(t):
            if t.dtype is torch.float32:
                return t.to(dtype=torch.bfloat16)
            return t

        if self.cfg.common.fp16:
            sample = utils.apply_to_sample(apply_half, sample)

        if self.cfg.common.bf16:
            sample = utils.apply_to_sample(apply_bfloat16, sample)

        if self._dummy_batch == "DUMMY":
            self._dummy_batch = sample

        return sample, False

    def _set_seed(self):
        # Set seed based on args.seed and the update number so that we get
        # reproducible results when resuming from checkpoints
        seed = self.cfg.common.seed + self.get_num_updates()
        utils.set_torch_seed(seed)

    def _sync_stats(self):
        # Return True if it's using multiple GPUs and DDP or multiple GPUs with
        # BMUF and it's a bmuf sync with warmup iterations completed before.
        if self.data_parallel_world_size == 1:
            return False
        elif self.cfg.optimization.use_bmuf:
            return (
                self.get_num_updates() + 1
            ) % self.cfg.bmuf.global_sync_iter == 0 and (
                self.get_num_updates() + 1
            ) > self.cfg.bmuf.warmup_iterations
        else:
            return True

    def _log_oom(self, exc):
        msg = "OOM: Ran out of memory with exception: {}".format(exc)
        logger.warning(msg)
        if torch.cuda.is_available() and hasattr(torch.cuda, "memory_summary"):
            for device_idx in range(torch.cuda.device_count()):
                logger.warning(torch.cuda.memory_summary(device=device_idx))
        sys.stderr.flush()

    def _aggregate_logging_outputs(
        self,
        logging_outputs: List[Dict[str, Any]],
        *extra_stats_to_sum,
        ignore=False,
    ):
        if self.task.__class__.logging_outputs_can_be_summed(self.get_criterion()):
            return self._fast_stat_sync_sum(
                logging_outputs, *extra_stats_to_sum, ignore=ignore
            )
        else:
            return self._all_gather_list_sync(
                logging_outputs, *extra_stats_to_sum, ignore=ignore
            )

    def _all_gather_list_sync(
        self,
        logging_outputs: List[Dict[str, Any]],
        *extra_stats_to_sum,
        ignore=False,
    ):
        """
        Sync logging outputs across workers. all_gather_list_sync is
        suitable when logging outputs are complex types.
        """
        if self.tpu:
            raise NotImplementedError
        if ignore:
            logging_outputs = []
        results = list(
            zip(
                *distributed_utils.all_gather_list(
                    [logging_outputs] + list(extra_stats_to_sum),
                    max_size=getattr(self.cfg.common, "all_gather_list_size", 16384),
                    group=self.data_parallel_process_group,
                )
            )
        )
        logging_outputs, extra_stats_to_sum = results[0], results[1:]
        logging_outputs = list(chain.from_iterable(logging_outputs))
        extra_stats_to_sum = [sum(s) for s in extra_stats_to_sum]
        return logging_outputs, extra_stats_to_sum

    def _fast_stat_sync_sum(
        self,
        logging_outputs: List[Dict[str, Any]],
        *extra_stats_to_sum,
        ignore=False,
    ):
        """
        Sync logging outputs across workers. fast_stat_sync_sum is
        faster than all_gather_list_sync, but is only suitable when
        logging outputs are scalars and can be summed. Note that
        *logging_outputs* cannot contain any nested dicts/lists.
        """
        data = {}
        for i, stat in enumerate(extra_stats_to_sum):
            data["extra_stats_" + str(i)] = stat
        if len(logging_outputs) > 0:
            log_keys = list(logging_outputs[0].keys())
            for k in log_keys:
                if not ignore:
                    v = sum(log[k] for log in logging_outputs if k in log)
                else:
                    v = logging_outputs[0][k]
                    v = torch.zeros_like(v) if torch.is_tensor(v) else 0
                data["logging_outputs_" + k] = v
        else:
            log_keys = None

        data = distributed_utils.all_reduce_dict(
            data, device=self.device, group=self.data_parallel_process_group
        )

        extra_stats_to_sum = [
            data["extra_stats_" + str(i)] for i in range(len(extra_stats_to_sum))
        ]
        if log_keys is not None:
            logging_outputs = [{k: data["logging_outputs_" + k] for k in log_keys}]
        else:
            logging_outputs = []
        return logging_outputs, extra_stats_to_sum

    def _check_grad_norms(self, grad_norm):
        """Check that grad norms are consistent across workers."""
        if self._grad_norm_buf is not None:
            self._grad_norm_buf.zero_()
            self._grad_norm_buf[self.data_parallel_rank] = grad_norm
            distributed_utils.all_reduce(
                self._grad_norm_buf, group=self.data_parallel_process_group
            )

            def is_consistent(tensor):
                max_abs_diff = torch.max(torch.abs(tensor - tensor[0]))
                return (
                    torch.isfinite(tensor).all()
                    or (max_abs_diff / (tensor[0] + 1e-6) < 1e-6).all()
                )

            if not is_consistent(self._grad_norm_buf):
                pretty_detail = "\n".join(
                    "rank {:3d} = {:.8f}".format(r, n)
                    for r, n in enumerate(self._grad_norm_buf.tolist())
                )
                error_detail = "grad_norm across the workers:\n{}\n".format(
                    pretty_detail
                )
                # use FloatingPointError to trigger NanDetector
                raise FloatingPointError(
                    "Fatal error: gradients are inconsistent between workers. "
                    "Try --ddp-backend=no_c10d. "
                    "Or are you mixing up different generation of GPUs in training?"
                    + "\n"
                    + "-" * 80
                    + "\n{}\n".format(error_detail)
                    + "-" * 80
                )

    def _reduce_and_log_stats(self, logging_outputs, sample_size, grad_norm=None):
        if grad_norm is not None and (
            not torch.is_tensor(grad_norm) or torch.isfinite(grad_norm)
        ):
            metrics.log_speed("ups", 1.0, priority=100, round=2)
            metrics.log_scalar("gnorm", grad_norm, priority=400, round=3)
            if self.cfg.optimization.clip_norm > 0:
                metrics.log_scalar(
                    "clip",
                    torch.where(
                        grad_norm > self.cfg.optimization.clip_norm,
                        grad_norm.new_tensor(100),
                        grad_norm.new_tensor(0),
                    ),
                    priority=500,
                    round=1,
                )

        with metrics.aggregate() as agg:
            if logging_outputs is not None:
                self.task.reduce_metrics(logging_outputs, self.get_criterion())
                del logging_outputs

            # extra warning for criterions that don't properly log a loss value
            if "loss" not in agg:
                if "loss" not in self._warn_once:
                    self._warn_once.add("loss")
                    logger.warning(
                        "Criterion.reduce_metrics did not log a 'loss' value, "
                        "which may break some functionality"
                    )
                metrics.log_scalar("loss", -1)

            # support legacy interface
            if self.tpu:
                logging_output = {}
            else:
                logging_output = agg.get_smoothed_values()
                logging_output["sample_size"] = sample_size
                for key_to_delete in ["ppl", "wps", "wpb", "bsz"]:
                    if key_to_delete in logging_output:
                        del logging_output[key_to_delete]
            return logging_output

    def _check_xla_compilation(self):
        import torch_xla.debug.metrics as met

        compile_stats = met.metric_data("CompileTime")
        if compile_stats is None:
            return
        num_xla_compiles = compile_stats[0]
        if num_xla_compiles > self._num_xla_compiles:
            logger.warning(
                "XLA compilation detected on device #{}; too many of these can lead "
                "to slow training, but we expect a few in the beginning".format(
                    self.cfg.distributed_training.distributed_rank
                )
            )
        self._num_xla_compiles = num_xla_compiles


def _catalog_shared_params(module, memo=None, prefix=""):
    if memo is None:
        first_call = True
        memo = {}
    else:
        first_call = False
    for name, param in module._parameters.items():
        param_prefix = prefix + ("." if prefix else "") + name
        if param not in memo:
            memo[param] = []
        memo[param].append(param_prefix)
    for name, m in module._modules.items():
        if m is None:
            continue
        submodule_prefix = prefix + ("." if prefix else "") + name
        _catalog_shared_params(m, memo, submodule_prefix)
    if first_call:
        return [x for x in memo.values() if len(x) > 1]


def _get_module_by_path(module, path):
    path = path.split(".")
    for name in path:
        module = getattr(module, name)
    return module


def _set_module_by_path(module, path, value):
    path = path.split(".")
    for name in path[:-1]:
        module = getattr(module, name)
    setattr(module, path[-1], value)

# train.py

In [22]:


def main(cfg: FairseqConfig) -> None: #학습 실행 코드
    if isinstance(cfg, argparse.Namespace):
        cfg = convert_namespace_to_omegaconf(cfg)

    utils.import_user_module(cfg.common)
    add_defaults(cfg)

    # 학습 분배 여부
    if (
        distributed_utils.is_master(cfg.distributed_training)
        and "job_logging_cfg" in cfg
    ):
        # make hydra logging work with ddp (see # see https://github.com/facebookresearch/hydra/issues/1126)
        logging.config.dictConfig(OmegaConf.to_container(cfg.job_logging_cfg))

    assert (
        cfg.dataset.max_tokens is not None or cfg.dataset.batch_size is not None
    ), "Must specify batch size either with --max-tokens or --batch-size"
    metrics.reset()

    # 로그 파일
    if cfg.common.log_file is not None:
        handler = logging.FileHandler(filename=cfg.common.log_file)
        logger.addHandler(handler)

    np.random.seed(cfg.common.seed) #랜덤 시드
    utils.set_torch_seed(cfg.common.seed)

    if distributed_utils.is_master(cfg.distributed_training):
        checkpoint_utils.verify_checkpoint_directory(cfg.checkpoint.save_dir)

    # Print args
    logger.info(cfg)

    # 체크포인트
    if cfg.checkpoint.write_checkpoints_asynchronously:
        try:
            import iopath  # noqa: F401
        except ImportError:
            logging.exception(
                "Asynchronous checkpoint writing is specified but iopath is "
                "not installed: `pip install iopath`"
            )
            return

    # Setup task, e.g., translation, language modeling, etc.
    task = tasks.setup_task(cfg.task) #어떤걸 할지 지정

    assert cfg.criterion, "Please specify criterion to train a model"

    # Build model and criterion
    if cfg.distributed_training.ddp_backend == "fully_sharded":
        with fsdp_enable_wrap(cfg.distributed_training):
            model = fsdp_wrap(task.build_model(cfg.model))
    else:
        model = task.build_model(cfg.model)
    criterion = task.build_criterion(cfg.criterion)
    logger.info(model)
    logger.info("task: {}".format(task.__class__.__name__))
    logger.info("model: {}".format(model.__class__.__name__))
    logger.info("criterion: {}".format(criterion.__class__.__name__))
    logger.info(
        "num. shared model params: {:,} (num. trained: {:,})".format(
            sum(
                p.numel() for p in model.parameters() if not getattr(p, "expert", False)
            ),
            sum(
                p.numel()
                for p in model.parameters()
                if not getattr(p, "expert", False) and p.requires_grad
            ),
        )
    )

    logger.info(
        "num. expert model params: {} (num. trained: {})".format(
            sum(p.numel() for p in model.parameters() if getattr(p, "expert", False)),
            sum(
                p.numel()
                for p in model.parameters()
                if getattr(p, "expert", False) and p.requires_grad
            ),
        )
    )

    # Load valid dataset (we load training data below, based on the latest checkpoint)
    # We load the valid dataset AFTER building the model
    if not cfg.dataset.disable_validation:
        data_utils.raise_if_valid_subsets_unintentionally_ignored(cfg)
        if cfg.dataset.combine_valid_subsets:
            task.load_dataset("valid", combine=True, epoch=1)
        else:
            for valid_sub_split in cfg.dataset.valid_subset.split(","):
                task.load_dataset(valid_sub_split, combine=False, epoch=1)

    # (optionally) Configure quantization
    if cfg.common.quantization_config_path is not None:
        quantizer = quantization_utils.Quantizer(
            config_path=cfg.common.quantization_config_path,
            max_epoch=cfg.optimization.max_epoch,
            max_update=cfg.optimization.max_update,
        )
    else:
        quantizer = None

    # Build trainer
    if cfg.common.model_parallel_size == 1:
        trainer = Trainer(cfg, task, model, criterion, quantizer)
    else:
        trainer = MegatronTrainer(cfg, task, model, criterion)
    logger.info(
        "training on {} devices (GPUs/TPUs)".format(
            cfg.distributed_training.distributed_world_size
        )
    )
    logger.info(
        "max tokens per device = {} and max sentences per device = {}".format(
            cfg.dataset.max_tokens,
            cfg.dataset.batch_size,
        )
    )

    # Load the latest checkpoint if one is available and restore the
    # corresponding train iterator
    extra_state, epoch_itr = checkpoint_utils.load_checkpoint(
        cfg.checkpoint,
        trainer,
        # don't cache epoch iterators for sharded datasets
        disable_iterator_cache=task.has_sharded_data("train"),
    )
    if cfg.common.tpu:
        import torch_xla.core.xla_model as xm

        xm.rendezvous("load_checkpoint")  # wait for all workers

    max_epoch = cfg.optimization.max_epoch or math.inf
    lr = trainer.get_lr()

    # TODO: a dry run on validation set to pin the memory
    valid_subsets = cfg.dataset.valid_subset.split(",")
    if not cfg.dataset.disable_validation:
        for subset in valid_subsets:
            logger.info('begin dry-run validation on "{}" subset'.format(subset))
            itr = trainer.get_valid_iterator(subset).next_epoch_itr(
                shuffle=False, set_dataset_epoch=False  # use a fixed valid set
            )
            if cfg.common.tpu:
                itr = utils.tpu_data_loader(itr)
            for _ in itr:
                pass
    # TODO: end of dry run section

    train_meter = meters.StopwatchMeter()
    train_meter.start()
    while epoch_itr.next_epoch_idx <= max_epoch:
        if lr <= cfg.optimization.stop_min_lr:
            logger.info(
                f"stopping training because current learning rate ({lr}) is smaller "
                "than or equal to minimum learning rate "
                f"(--stop-min-lr={cfg.optimization.stop_min_lr})"
            )
            break

        # train for one epoch
        valid_losses, should_stop = train(cfg, trainer, task, epoch_itr)
        if should_stop:
            break

        # only use first validation loss to update the learning rate
        lr = trainer.lr_step(epoch_itr.epoch, valid_losses[0])

        epoch_itr = trainer.get_train_iterator(
            epoch_itr.next_epoch_idx,
            # sharded data: get train iterator for next epoch
            load_dataset=task.has_sharded_data("train"),
            # don't cache epoch iterators for sharded datasets
            disable_iterator_cache=task.has_sharded_data("train"),
        )
    train_meter.stop()
    logger.info("done training in {:.1f} seconds".format(train_meter.sum))

    # ioPath implementation to wait for all asynchronous file writes to complete.
    if cfg.checkpoint.write_checkpoints_asynchronously:
        logger.info(
            "ioPath PathManager waiting for all asynchronous checkpoint "
            "writes to finish."
        )
        PathManager.async_close()
        logger.info("ioPath PathManager finished waiting.")


def should_stop_early(cfg: DictConfig, valid_loss: float) -> bool:
    # skip check if no validation was done in the current epoch
    if valid_loss is None:
        return False
    if cfg.checkpoint.patience <= 0:
        return False

    def is_better(a, b):
        return a > b if cfg.checkpoint.maximize_best_checkpoint_metric else a < b

    prev_best = getattr(should_stop_early, "best", None)
    if prev_best is None or is_better(valid_loss, prev_best):
        should_stop_early.best = valid_loss
        should_stop_early.num_runs = 0
        return False
    else:
        should_stop_early.num_runs += 1
        if should_stop_early.num_runs >= cfg.checkpoint.patience:
            logger.info(
                "early stop since valid performance hasn't improved for last {} runs".format(
                    cfg.checkpoint.patience
                )
            )
            return True
        else:
            return False


@metrics.aggregate("train")
def train(
    cfg: DictConfig, trainer: Trainer, task: tasks.FairseqTask, epoch_itr
) -> Tuple[List[Optional[float]], bool]:
    """Train the model for one epoch and return validation losses."""
    # Initialize data iterator
    itr = epoch_itr.next_epoch_itr(
        fix_batches_to_gpus=cfg.distributed_training.fix_batches_to_gpus,
        shuffle=(epoch_itr.next_epoch_idx > cfg.dataset.curriculum),
    )
    update_freq = (
        cfg.optimization.update_freq[epoch_itr.epoch - 1]
        if epoch_itr.epoch <= len(cfg.optimization.update_freq)
        else cfg.optimization.update_freq[-1]
    )
    itr = iterators.GroupedIterator(
        itr,
        update_freq,
        skip_remainder_batch=cfg.optimization.skip_remainder_batch,
    )
    if cfg.common.tpu:
        itr = utils.tpu_data_loader(itr)
    progress = progress_bar.progress_bar(
        itr,
        log_format=cfg.common.log_format,
        log_file=cfg.common.log_file,
        log_interval=cfg.common.log_interval,
        epoch=epoch_itr.epoch,
        aim_repo=(
            cfg.common.aim_repo
            if distributed_utils.is_master(cfg.distributed_training)
            else None
        ),
        aim_run_hash=(
            cfg.common.aim_run_hash
            if distributed_utils.is_master(cfg.distributed_training)
            else None
        ),
        aim_param_checkpoint_dir=cfg.checkpoint.save_dir,
        tensorboard_logdir=(
            cfg.common.tensorboard_logdir
            if distributed_utils.is_master(cfg.distributed_training)
            else None
        ),
        default_log_format=("tqdm" if not cfg.common.no_progress_bar else "simple"),
        wandb_project=(
            cfg.common.wandb_project
            if distributed_utils.is_master(cfg.distributed_training)
            else None
        ),
        wandb_run_name=os.environ.get(
            "WANDB_NAME", os.path.basename(cfg.checkpoint.save_dir)
        ),
        azureml_logging=(
            cfg.common.azureml_logging
            if distributed_utils.is_master(cfg.distributed_training)
            else False
        ),
    )
    progress.update_config(_flatten_config(cfg))

    trainer.begin_epoch(epoch_itr.epoch)

    valid_subsets = cfg.dataset.valid_subset.split(",")
    should_stop = False
    num_updates = trainer.get_num_updates()
    logger.info("Start iterating over samples")
    for i, samples in enumerate(progress):
        with metrics.aggregate("train_inner"), torch.autograd.profiler.record_function(
            "train_step-%d" % i
        ):
            log_output = trainer.train_step(samples)

        if log_output is not None:  # not OOM, overflow, ...
            # log mid-epoch stats
            num_updates = trainer.get_num_updates()
            if num_updates % cfg.common.log_interval == 0:
                stats = get_training_stats(metrics.get_smoothed_values("train_inner"))
                progress.log(stats, tag="train_inner", step=num_updates)

                # reset mid-epoch stats after each log interval
                # the end-of-epoch stats will still be preserved
                metrics.reset_meters("train_inner")

        end_of_epoch = not itr.has_next()
        valid_losses, should_stop = validate_and_save(
            cfg, trainer, task, epoch_itr, valid_subsets, end_of_epoch
        )

        if should_stop:
            break

    # log end-of-epoch stats
    logger.info("end of epoch {} (average epoch stats below)".format(epoch_itr.epoch))
    stats = get_training_stats(metrics.get_smoothed_values("train"))
    progress.print(stats, tag="train", step=num_updates)

    # reset epoch-level meters
    metrics.reset_meters("train")
    return valid_losses, should_stop


def _flatten_config(cfg: DictConfig):
    config = OmegaConf.to_container(cfg)
    # remove any legacy Namespaces and replace with a single "args"
    namespace = None
    for k, v in list(config.items()):
        if isinstance(v, argparse.Namespace):
            namespace = v
            del config[k]
    if namespace is not None:
        config["args"] = vars(namespace)
    return config


def validate_and_save(
    cfg: DictConfig,
    trainer: Trainer,
    task: tasks.FairseqTask,
    epoch_itr,
    valid_subsets: List[str],
    end_of_epoch: bool,
) -> Tuple[List[Optional[float]], bool]:
    num_updates = trainer.get_num_updates()
    max_update = cfg.optimization.max_update or math.inf

    # Stopping conditions (and an additional one based on validation loss later
    # on)
    should_stop = False
    if num_updates >= max_update:
        should_stop = True
        logger.info(
            f"Stopping training due to "
            f"num_updates: {num_updates} >= max_update: {max_update}"
        )

    training_time_hours = trainer.cumulative_training_time() / (60 * 60)
    if (
        cfg.optimization.stop_time_hours > 0
        and training_time_hours > cfg.optimization.stop_time_hours
    ):
        should_stop = True
        logger.info(
            f"Stopping training due to "
            f"cumulative_training_time: {training_time_hours} > "
            f"stop_time_hours: {cfg.optimization.stop_time_hours} hour(s)"
        )

    do_save = ( #저장, 일정 간격으로
        (end_of_epoch and epoch_itr.epoch % cfg.checkpoint.save_interval == 0)
        or should_stop
        or (
            cfg.checkpoint.save_interval_updates > 0
            and num_updates > 0
            and num_updates % cfg.checkpoint.save_interval_updates == 0
            and num_updates >= cfg.dataset.validate_after_updates
        )
    )
    do_validate = (# 검증, 일정 간격으로
        (
            (not end_of_epoch and do_save)  # validate during mid-epoch saves
            or (end_of_epoch and epoch_itr.epoch % cfg.dataset.validate_interval == 0)
            or should_stop
            or (
                cfg.dataset.validate_interval_updates > 0
                and num_updates > 0
                and num_updates % cfg.dataset.validate_interval_updates == 0
            )
        )
        and not cfg.dataset.disable_validation
        and num_updates >= cfg.dataset.validate_after_updates
    )

    # Validate
    valid_losses = [None] #검증하면서 로스값 계산
    if do_validate:
        valid_losses = validate(cfg, trainer, task, epoch_itr, valid_subsets)

    should_stop |= should_stop_early(cfg, valid_losses[0])

    # Save checkpoint
    if do_save or should_stop:
        cp_path = checkpoint_utils.save_checkpoint(
            cfg.checkpoint, trainer, epoch_itr, valid_losses[0]
        )
        if cp_path is not None and hasattr(task, "post_save"):
            task.post_save(cp_path, num_updates)

    return valid_losses, should_stop


def get_training_stats(stats: Dict[str, Any]) -> Dict[str, Any]:
    stats["wall"] = round(metrics.get_meter("default", "wall").elapsed_time, 0)
    return stats


def validate( #검증 코드
    cfg: DictConfig,
    trainer: Trainer,
    task: tasks.FairseqTask,
    epoch_itr,
    subsets: List[str],
) -> List[Optional[float]]:
    """Evaluate the model on the validation set(s) and return the losses."""

    if cfg.dataset.fixed_validation_seed is not None:
        # set fixed seed for every validation
        utils.set_torch_seed(cfg.dataset.fixed_validation_seed)

    trainer.begin_valid_epoch(epoch_itr.epoch)
    valid_losses = []
    for subset_idx, subset in enumerate(subsets): #데이터셋을 서브셋만큼 나눠 검증 진행
        logger.info('begin validation on "{}" subset'.format(subset))

        # Initialize data iterator
        itr = trainer.get_valid_iterator(subset).next_epoch_itr(
            shuffle=False, set_dataset_epoch=False  # use a fixed valid set
        )
        if cfg.common.tpu:
            itr = utils.tpu_data_loader(itr)
        progress = progress_bar.progress_bar(
            itr,
            log_format=cfg.common.log_format,
            log_interval=cfg.common.log_interval,
            epoch=epoch_itr.epoch,
            prefix=f"valid on '{subset}' subset",
            aim_repo=(
                cfg.common.aim_repo
                if distributed_utils.is_master(cfg.distributed_training)
                else None
            ),
            aim_run_hash=(
                cfg.common.aim_run_hash
                if distributed_utils.is_master(cfg.distributed_training)
                else None
            ),
            aim_param_checkpoint_dir=cfg.checkpoint.save_dir,
            tensorboard_logdir=(
                cfg.common.tensorboard_logdir
                if distributed_utils.is_master(cfg.distributed_training)
                else None
            ),
            default_log_format=("tqdm" if not cfg.common.no_progress_bar else "simple"),
            wandb_project=(
                cfg.common.wandb_project
                if distributed_utils.is_master(cfg.distributed_training)
                else None
            ),
            wandb_run_name=os.environ.get(
                "WANDB_NAME", os.path.basename(cfg.checkpoint.save_dir)
            ),
        )

        # create a new root metrics aggregator so validation metrics
        # don't pollute other aggregators (e.g., train meters)
        with metrics.aggregate(new_root=True) as agg:
            for i, sample in enumerate(progress):
                if (
                    cfg.dataset.max_valid_steps is not None
                    and i > cfg.dataset.max_valid_steps
                ):
                    break
                trainer.valid_step(sample)

        # log validation stats
        # only tracking the best metric on the 1st validation subset
        tracking_best = subset_idx == 0
        stats = get_valid_stats(cfg, trainer, agg.get_smoothed_values(), tracking_best)

        if hasattr(task, "post_validate"):
            task.post_validate(trainer.get_model(), stats, agg)

        progress.print(stats, tag=subset, step=trainer.get_num_updates())

        valid_losses.append(stats[cfg.checkpoint.best_checkpoint_metric])
    return valid_losses


def get_valid_stats(
    cfg: DictConfig,
    trainer: Trainer,
    stats: Dict[str, Any],
    tracking_best: bool,
) -> Dict[str, Any]:
    stats["num_updates"] = trainer.get_num_updates()
    if tracking_best and hasattr(checkpoint_utils.save_checkpoint, "best"):
        key = "best_{0}".format(cfg.checkpoint.best_checkpoint_metric)
        best_function = max if cfg.checkpoint.maximize_best_checkpoint_metric else min
        stats[key] = best_function(
            checkpoint_utils.save_checkpoint.best,
            stats[cfg.checkpoint.best_checkpoint_metric],
        )
    return stats


def cli_main(
    modify_parser: Optional[Callable[[argparse.ArgumentParser], None]] = None
) -> None:
    parser = options.get_training_parser()
    args = options.parse_args_and_arch(parser, modify_parser=modify_parser)

    cfg = convert_namespace_to_omegaconf(args)

    if cfg.common.use_plasma_view:
        server = PlasmaStore(path=cfg.common.plasma_path)
        logger.info(
            f"Started plasma server pid {server.server.pid} {cfg.common.plasma_path}"
        )

    if args.profile:
        with torch.cuda.profiler.profile():
            with torch.autograd.profiler.emit_nvtx():
                distributed_utils.call_main(cfg, main)
    else:
        distributed_utils.call_main(cfg, main)

    # if cfg.common.use_plasma_view:
    #     server.server.kill()




#사전 준비 작업

In [ ]:
#스크립트 생성

def load_json(temp_path):
    wav_list = list()
    text_list = list()
    with open(temp_path) as f:
        data_list=json.load(f)
        for temp_data in data_list:
            wav = temp_data['wav']
            text = temp_data['text']
            wav_list.append(wav)
            text_list.append(text)
    return wav_list, text_list

## Fairseq 스타일로 변환하기
def get_parser():
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--root", default='/code/gitRepo/data/clovacall', metavar="DIR",
        help="root directory containing flac files to index"
    )
    parser.add_argument(
        "--dest", default='/code/gitRepo/data/clovacall/script', type=str, metavar="DIR",
        help="output directory"
    )
    parser.add_argument(
        "--dev_portion", default=600, type=int,
        help="dev portion"
    )
    return parser


def load_audio(file_path):
    ext = Path(file_path).suffix

    if ext in ['.wav', '.flac']:
        wav, sr = librosa.load(file_path, sr=16000)
    elif ext == '.pcm':
        wav = np.memmap(file_path, dtype='h', mode='r').astype('float32') / 32767
        sr = 16000
    elif ext in ['.raw', '.RAW']:
        wav, sr = sf.read(file_path, channels=1, samplerate=16000,
                          format='RAW', subtype='PCM_16')
    else:
        raise ValueError("Unsupported preprocess method : {0}".format(Path(file_path).suffix))

    return wav, sr



def make_script(args, dir_path, wav_list, text_list):
    fileinfo = list()
    except_files = list()

    ## 숫자, 영어, 한글을 제외하고 특수문자 전체 제거(.?/)
    pattern = '[^\w\s]'

    for audio_path, raw_sentence in zip(wav_list, text_list):
        audio_path = os.path.join(dir_path, audio_path)
        audio_path = os.path.realpath(audio_path)

        try:
            wav, sr = load_audio(audio_path)

            ## 길이
            new_sentence = re.sub(pattern=pattern, repl='', string=raw_sentence).strip()
            fileinfo.append("{} :: {}".format(os.path.relpath(audio_path, args.root), new_sentence))

        except:
            except_files.append(audio_path)

    return fileinfo, except_files

def save_trn(args, fileinfo, file_name='train'):

    print("save files [{}]".format(file_name))
    with open(os.path.join(args.dest, "{}.trn".format(file_name)), 'w', encoding='UTF8') as trn_out:
        for trn_item in fileinfo:
            print(trn_item, file=trn_out)

def main(args):
    ## 데이터 형태가 반드시 일치해야 합니다.
    """
    |-clovacall
         |------train_ClovaCall.json
         |------test_ClovaCall.json
         |------wavs_train
                    |----------41_0514_688_0_07118_05.wav
                    |----------41_0509_714_0_08568_02.wav
         |------wavs_test
                    |----------41_0514_301_0_07111_09.wav
                    |----------41_0515_577_0_04088_07.wav
    """

    train_json_name = 'train_ClovaCall.json'
    test_json_name = 'test_ClovaCall.json'
    train_folder_name = 'wavs_train'
    test_folder_name = 'wavs_test'

    assert os.path.isdir(args.root), "폴더가 없습니다. 다시 한번 확인해 주세요 [{}]".format(args.root)
    folder_list = os.listdir(args.root)
    for item in [train_json_name, test_json_name, train_folder_name, test_folder_name]:
        assert item in folder_list, "파일이 없습니다. 다시 한번 확인해 주세요. [{}]".format(item)

    os.makedirs(args.dest, exist_ok=True)

    train_wav, train_text = load_json(os.path.join(args.root, train_json_name))
    train_wav, valid_wav, train_text, valid_text = train_test_split(train_wav, train_text, test_size=args.dev_portion)
    test_wav, test_text = load_json(os.path.join(args.root, test_json_name))


    train_folder= os.path.join(args.root, train_folder_name)
    fileinfo, except_files = make_script(args, train_folder, train_wav, train_text)
    save_trn(args, fileinfo, 'train')
    print("저장된 파일", len(fileinfo))
    print("제외된 파일", len(except_files))

    dev_folder = os.path.join(args.root, train_folder_name)
    fileinfo, except_files = make_script(args, dev_folder, valid_wav, valid_text)
    save_trn(args, fileinfo, 'dev')
    print("저장된 파일", len(fileinfo))
    print("제외된 파일", len(except_files))

    test_folder = os.path.join(args.root, test_folder_name)
    fileinfo, except_files = make_script(args, test_folder, test_wav, test_text)
    save_trn(args, fileinfo, 'eval_clean')
    print("저장된 파일", len(fileinfo))
    print("제외된 파일", len(except_files))

    return


if __name__ == '__main__':
    parser = get_parser()
    args = parser.parse_args()


    def _print_config(config):
        import pprint
        pp = pprint.PrettyPrinter(indent=4)
        pp.pprint(vars(config))

    _print_config(args)

    main(args)

In [ ]:
# yaml 생성

## Fairseq 스타일로 변환하기
def get_parser():
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--root", default='/code/gitRepo/data/aihub/ksponspeech', metavar="DIR",
        help="root directory containing flac files to index"
    )

    parser.add_argument(
        "--info", default=None, metavar="DIR",
        help="전처리 추가적으로 수행한 것."
    )
    parser.add_argument(
        "--do_info", action="store_true",
        help="전처리 추가적으로 수행할지 여부 확인"
    )
    parser.add_argument(
        "--do_remove", action="store_true",
        help="한글 음소가 아닌 숫자, 영어가 포함되어 있는 모든 단어를 삭제할지 여부 확인"
    )
    parser.add_argument(
        "--token_limit", default=sys.maxsize, type=int,
        help="최대 글자수 체크"
    )

    parser.add_argument(
        "--dest", default='manifest_temp', type=str, metavar="DIR", help="output directory"
    )
    parser.add_argument(
        "--ext", default="pcm", type=str, metavar="EXT", help="extension to look for"
    )
    parser.add_argument('--preprocess_mode', type=str,
                        default='phonetic',
                        help='Ex) (70%)/(칠 십 퍼센트) 확률이라니 (뭐 뭔)/(모 몬) 소리야 진짜 (100%)/(백 프로)가 왜 안돼?'
                             'phonetic: 칠 십 퍼센트 확률이라니 모 몬 소리야 진짜 백 프로가 왜 안돼?'
                             'spelling: 70% 확률이라니 뭐 뭔 소리야 진짜 100%가 왜 안돼?')
    parser.add_argument('--output_unit', type=str,
                        default='grapheme',
                        help='character or subword or grapheme')
    parser.add_argument('--additional_output_unit', type=str,
                        default=None,
                        help='character or subword or grapheme')
    parser.add_argument("--seed", default=42, type=int, metavar="N", help="random seed")
    parser.add_argument(
        "--time",
        default=None,
        type=str,
        metavar="MIN",
        help="set if you want make split manifest",
    )
    parser.add_argument('--script_path', type=str,
                        default="/code/gitRepo/data/aihub/ksponspeech/KsponSpeech_scripts",
                        help='AIHUB에서 제공해 주는 스크립트 폴더')
    parser.add_argument(
        "--del_silence", action="store_true",
        help="음성이 없는 곳을 삭제하는 건 어때?"
    )
    return parser


def find_index(durations, limit): #인덱스 찾기
    for idx in range(len(durations)):
        if sum(durations[:idx]) > limit:
            return idx
    return len(durations)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)


def load_yaml(yaml_path):
    # Read YAML file
    with open(yaml_path, 'r') as stream:
        data_loaded = yaml.load(stream, Loader=yaml.FullLoader)
    return data_loaded


def load_info(info_path):
    if not os.path.isdir(info_path):
        return {}

    info_files = [filename for filename in os.listdir(info_path) if '.yaml' in filename]
    info_data = {}
    for filename in info_files:
        file_path = os.path.join(info_path, filename)
        temp_data = load_yaml(file_path)
        info_data.update(temp_data)
    return info_data

def save_converted_info(args, name, converted_info):
    if len(converted_info) == 0:
        return

    yaml_dict = {k: v for k, v in sorted(converted_info.items(), key=lambda item: (len(item[0]), item[0]))}
    with open(os.path.join(args.dest, '{}.yaml'.format(name)), 'w', encoding="utf-8") as write_f:
        yaml.dump(yaml_dict, write_f, allow_unicode=True, default_style=None, default_flow_style=False)


def save_wrong_script(args, name, transcripts, fileinfo, raw_sentences, new_sentences):
    ## 틀린 것 저장하기

    ## 알파벳 추가
    reg = re.compile(r'[A-Z]')
    yaml_dict = {}
    for grapheme_transcript, fileitem, raw_sentence, new_sentence in zip(transcripts, fileinfo, raw_sentences,
                                                                         new_sentences):
        graphemes = grapheme_transcript.split()
        file_num = Path(fileitem.split()[0]).stem.split("_")[1]
        assert len(file_num) == 6

        for grapheme in graphemes:
            if grapheme.isdigit() or reg.match(grapheme):
                yaml_dict[file_num] = str(raw_sentence.replace('\n', ''))

    if len(yaml_dict) == 0:
        return

    ## Sorting
    yaml_dict = {k: v for k, v in sorted(yaml_dict.items(), key=lambda item: (len(item[0]), item[0]))}
    with open(os.path.join(args.dest, '{}.yaml'.format(name)), 'w', encoding="utf-8") as write_f:
        yaml.dump(yaml_dict, write_f, allow_unicode=True, default_style=None, default_flow_style=False)


def save_dict(args, transcripts, dict_name='dict.ltr.txt', alphabet_name='alphabet.txt'):
    vocab_list = list()
    vocab_freq = list()
    for grapheme_transcript in transcripts:
        graphemes = grapheme_transcript.split()

        for grapheme in graphemes:
            if grapheme not in vocab_list:
                vocab_list.append(grapheme)
                vocab_freq.append(1)
            else:
                vocab_freq[vocab_list.index(grapheme)] += 1

    ## write ltr
    vocab_freq, vocab_list = zip(*sorted(zip(vocab_freq, vocab_list), reverse=True))
    with open(os.path.join(args.dest, dict_name), 'w') as write_f:
        for idx, (grpm, freq) in enumerate(zip(vocab_list, vocab_freq)):
            print("{} {}".format(grpm, freq), file=write_f)

    ## Write Vocab files
    with open(os.path.join(args.dest, alphabet_name), 'w', encoding='UTF8') as write_f:
        print("# Each line in this file represents the Unicode codepoint (UTF-8 encoded)", file=write_f)
        print("# associated with a numeric label.", file=write_f)
        print("# A line that starts with # is a comment. You can escape it with \# if you wish", file=write_f)
        print("# to use '#' as a label.", file=write_f)
        for token in vocab_list:
            print(token, file=write_f)
        ## final token must be \n
        print('', file=write_f)

        print("# The last (non-comment) line needs to end with a newline.", file=write_f, end='')

    return

def save_lexicon(args, texts, lexicon_name='lexicon.lst'):
    vocab_list = {}
    for text in texts:
        for word in text.split():
            new_word = word + "|"
            vocab_list[word] = " ".join(new_word)

    ## Write Vocab files
    ## Sorting
    vocab_list = {k: v for k, v in sorted(vocab_list.items(), key=lambda item: item[0])}
    with open(os.path.join(args.dest, lexicon_name), 'w', encoding='UTF8') as write_f:
        for k, v in vocab_list.items():
            print("{}\t{}".format(k,v), file=write_f)
    return




def save_files(args, file_name, dir_path, fileinfo, texts, transcripts):
    with open(os.path.join(args.dest, file_name + ".tsv"), 'w') as tsv_out, open(
            os.path.join(args.dest, file_name + ".ltr"), "w"
    ) as ltr_out, open(
        os.path.join(args.dest, file_name + ".wrd"), "w"
    ) as wrd_out:

        print(dir_path, file=tsv_out)
        for tsv_item, wrd_item, ltr_item in zip(fileinfo, texts, transcripts):
            print(tsv_item, file=tsv_out)
            print(wrd_item, file=wrd_out)
            print(ltr_item + " |", file=ltr_out)

    print("save files [{}]".format(file_name))
    return



def pcm2wav(pcm_file, channels=1, bit_depth=16, sampling_rate=16000):
    wav_file = str(Path(pcm_file).with_suffix('.wav'))
    # Check if the options are valid.
    if bit_depth % 8 != 0:
        raise ValueError("bit_depth " + str(bit_depth) + " must be a multiple of 8.")

    # Read the .pcm file as a binary file and store the data to pcm_data
    with open(pcm_file, 'rb') as opened_pcm_file:
        pcm_data = opened_pcm_file.read()
        with wave.open(wav_file, 'wb') as obj2write:
            obj2write.setnchannels(channels)
            obj2write.setsampwidth(bit_depth // 8)
            obj2write.setframerate(sampling_rate)
            obj2write.writeframes(pcm_data)

    return wav_file

def load_script(args, script_path, info_data, token_limit=sys.maxsize):
    assert os.path.isfile(script_path)

    fileinfo = list()
    durations = list()
    texts = list()
    audio_nums = list()
    transcripts = list()

    additional_texts = list()
    additional_transcripts = list()

    raw_sentences = list()
    new_sentences = list()

    converted_info = {}

    reg = re.compile(r'.*[a-zA-Z0-9]')
    limit_count = 0
    remove_count = 0
    with open(script_path, "r") as f:
        for line in tqdm(f):
            convert_flag = False

            items = line.split(" :: ")
            file_path = os.path.join(args.root, items[0])
            file_path = os.path.realpath(file_path)
            audio_num = str(Path(file_path).stem.split("_")[1])
            raw_sentence = items[1]
            if len(audio_num) ==6 and audio_num in info_data:
                raw_sentence = info_data[audio_num]
                convert_flag=True

            ## 확장자 확인
            if args.ext == 'pcm':
                try:
                    wav = np.memmap(file_path, dtype='h', mode='r').astype('float32') / 32767
                    sr = 16000
                except ValueError:
                    # print('pcm load 에러 wave로 교체 [{}]'.format(file_path))
                    file_path = pcm2wav(file_path)
                    wav, sr = librosa.load(file_path, sr=16000)

            elif args.ext in ['flac', 'wav']:
                wav, sr = librosa.load(file_path, sr=16000)
            else:
                raise ValueError("Unsupported extention method : {0}".format(args.ext))

            if args.del_silence:
                non_silence_indices = librosa.effects.split(wav, top_db=30)
                wav = np.concatenate([wav[start:end] for start, end in non_silence_indices])
            frames = len(wav)

            if len(audio_num) ==6:
                new_sentence = preprocess(raw_sentence=raw_sentence, mode=args.preprocess_mode, audio_num=audio_num)
            else:
                new_sentence = raw_sentence.replace('\n', '')

            ##################################
            if len(new_sentence) > token_limit:
                limit_count+=1
                continue

            if args.do_remove and reg.match(new_sentence) and args.preprocess_mode != 'spelling':
                converted_info[audio_num] = new_sentence
                remove_count += 1
                continue
            #################################


            ## 저장 모드는 여기에 추가하기.
            if args.output_unit == 'grapheme':
                texts.append(unicodedata.normalize('NFKD', new_sentence).upper())
                transcripts.append(" ".join(unicodedata.normalize('NFKD', new_sentence).replace(' ', '|')).upper())
            elif args.output_unit == 'character':
                texts.append(new_sentence.upper())
                transcripts.append(" ".join(list(new_sentence.replace(' ', '|').upper())))
            else:
                raise ValueError("Unsupported preprocess method : {0}".format(args.output_unit))

            ## 저장 모드는 여기에 추가하기.
            if args.additional_output_unit is not None:
                if args.additional_output_unit == 'grapheme':
                    additional_texts.append(unicodedata.normalize('NFKD', new_sentence).upper())
                    additional_transcripts.append(" ".join(unicodedata.normalize('NFKD', new_sentence).replace(' ', '|')).upper())
                elif args.additional_output_unit == 'character':
                    additional_texts.append(new_sentence.upper())
                    additional_transcripts.append(" ".join(list(new_sentence.replace(' ', '|').upper())))
                else:
                    raise ValueError("Unsupported preprocess method : {0}".format(args.output_unit))

            if convert_flag:
                converted_info[audio_num] = new_sentence

            ## 넣기
            fileinfo.append("{}\t{}".format(os.path.relpath(file_path, args.root), frames))
            durations.append(frames)
            audio_nums.append(audio_num)
            raw_sentences.append(raw_sentence)
            new_sentences.append(new_sentence)
    print("총 무시된 숫자 : ", limit_count+remove_count)
    print("길이를 넘겨서 무시된 숫자 : ", limit_count)
    print("숫자등이 있어서 무시된 숫자 : ", remove_count)

    return fileinfo, durations, texts, audio_nums, transcripts, raw_sentences, new_sentences, converted_info, additional_texts, additional_transcripts



def main(args):
    if not os.path.exists(args.dest):
        os.makedirs(args.dest)
    args.root = os.path.realpath(args.root)

    ## --dataset_path 에 있어야 하는 폴더들
    #for folder in ['KsponSpeech_01','KsponSpeech_02','KsponSpeech_03','KsponSpeech_04','KsponSpeech_05','KsponSpeech_eval']:
    #    if folder not in os.listdir(args.root):
    #        assert os.path.isdir(folder), "root 위치에 해당 폴더가 반드시 필요합니다. [{}]".format(folder)

    assert os.path.isdir(args.script_path), "aihub에서 제공해주는 스크립트 폴더를 넣어주시기 바랍니다. script_path : [{}]".format(args.script_path)

    ## Info 파일 불러오기
    info_data = {}
    if args.do_info:
        ## info 파일 불러오기
        info_data = load_info(args.info)

    ## .trn 확장자만 확인함
    file_list = [file for file in os.listdir(args.script_path) if Path(file).suffix == '.trn']
    assert len(file_list) > 0, "스크립트 파일이 한개도 없네요 [{}]".format(args.script_path)

    ## 스크립트 읽어오기.
    script_name = 'train.trn'
    if script_name in file_list:
        print("generate [{}]".format(script_name))
        fileinfo, durations, texts, audio_nums, transcripts, raw_sentences, new_sentences, converted_info,  additional_texts, additional_transcripts = load_script(args, os.path.join(args.script_path, script_name), info_data, token_limit=args.token_limit)
        fileinfo = np.array(fileinfo)
        durations = np.array(durations)
        texts = np.array(texts)
        transcripts = np.array(transcripts)

        ## 추가용
        additional_texts = np.array(additional_texts)
        additional_transcripts = np.array(additional_transcripts)

        ## lexicon 만들기
        save_lexicon(args, texts, lexicon_name='lexicon.lst')
        ## dictionary 저장
        save_dict(args, transcripts, dict_name='dict.ltr.txt', alphabet_name='alphabet.txt')

        ## 추가용 만들기
        if args.additional_output_unit is not None:
            ## lexicon 만들기
            save_lexicon(args, additional_texts, lexicon_name='add_lexicon.lst')
            ## dictionary 저장
            save_dict(args, additional_transcripts, dict_name='add_dict.ltr.txt', alphabet_name='add_alphabet.txt')

        #save_wrong_script(args, 'train_wrong',transcripts, fileinfo, raw_sentences, new_sentences)
        save_converted_info(args, 'train_converted', converted_info)

        ## train 이랑 dev 나눠서 저장
        train_ids = [idx for idx, num in enumerate(audio_nums)]
        limit_idx = len(train_ids)
        if args.time is not None:
            random.shuffle(train_ids)
            assert args.time in ['10min', '1hour', '10hour', '100hour'], '설정 재대로 해라...'
            time_limit = 0
            if args.time == '10min':
                ## 16000 hz * 60초 * 10분
                time_limit = 16000 * 60 * 10
            if args.time == '1hour':
                ## 16000 hz * 60초 * 60분 * 1
                time_limit = 16000 * 60 * 60 * 1
            if args.time == '10hour':
                ## 16000 hz * 60초 * 60분 * 10
                time_limit = 16000 * 60 * 60 * 10
            if args.time == '100hour':
                ## 16000 hz * 60초 * 60분 * 100
                time_limit = 16000 * 60 * 60 * 100

            limit_idx = find_index(durations[train_ids], time_limit)

        save_files(args, 'train', args.root, fileinfo[train_ids[:limit_idx]], texts[train_ids[:limit_idx]],
                   transcripts[train_ids[:limit_idx]])
        ## 추가용 만들기
        if args.additional_output_unit is not None:
            save_files(args, 'add_train', args.root, fileinfo[train_ids[:limit_idx]], additional_texts[train_ids[:limit_idx]],
                       additional_transcripts[train_ids[:limit_idx]])

    ## 스크립트 읽어오기.
    script_name = 'dev.trn'
    if script_name in file_list:
        print("generate [{}]".format(script_name))
        fileinfo, durations, texts, audio_nums, transcripts, raw_sentences, new_sentences, converted_info, additional_texts, additional_transcripts = load_script(args, os.path.join(args.script_path, script_name), info_data)
        save_files(args, 'dev', args.root, fileinfo, texts, transcripts)

        ## 추가용 만들기
        if args.additional_output_unit is not None:
            save_files(args, 'add_dev', args.root, fileinfo, additional_texts, additional_transcripts)

        #save_wrong_script(args, 'dev_wrong', transcripts, fileinfo, raw_sentences, new_sentences)
        save_converted_info(args, 'dev_converted', converted_info)

    ## 스크립트 읽어오기.
    script_name = 'eval_other.trn'
    if script_name in file_list:
        print("generate [{}]".format(script_name))
        fileinfo, durations, texts, audio_nums, transcripts, raw_sentences, new_sentences, converted_info, additional_texts, additional_transcripts = load_script(args, os.path.join(args.script_path,
                                                                                             script_name), info_data)
        save_files(args, 'eval_other', args.root, fileinfo, texts, transcripts)

        ## 추가용 만들기
        if args.additional_output_unit is not None:
            save_files(args, 'add_eval_other', args.root, fileinfo, additional_texts, additional_transcripts)

        #save_wrong_script(args, 'eval_other_wrong', transcripts, fileinfo, raw_sentences, new_sentences)
        save_converted_info(args, 'eval_other_converted', converted_info)

    ## 스크립트 읽어오기.
    script_name = 'eval_clean.trn'
    if script_name in file_list:
        print("generate [{}]".format(script_name))
        fileinfo, durations, texts, audio_nums, transcripts, raw_sentences, new_sentences, converted_info, additional_texts, additional_transcripts = load_script(args, os.path.join(args.script_path,
                                                                                             script_name), info_data)
        save_files(args, 'eval_clean', args.root, fileinfo, texts, transcripts)

        ## 추가용 만들기
        if args.additional_output_unit is not None:
            save_files(args, 'add_eval_clean', args.root, fileinfo, additional_texts, additional_transcripts)
        #save_wrong_script(args, 'eval_clean_wrong', transcripts, fileinfo, raw_sentences, new_sentences)
        save_converted_info(args, 'eval_clean_converted', converted_info)


if __name__ == '__main__':
    parser = get_parser()
    args = parser.parse_args()


    def _print_config(config):
        import pprint
        pp = pprint.PrettyPrinter(indent=4)
        pp.pprint(vars(config))

    _print_config(args)

    main(args)

In [ ]:
#전처리 합쳐서 하기

def bracket_filter(sentence, mode='phonetic'):
    new_sentence = str()

    if mode == 'phonetic':
        flag = False

        for ch in sentence:
            if ch == '(' and flag is False:
                flag = True
                continue
            if ch == '(' and flag is True:
                flag = False
                continue
            if ch != ')' and flag is False:
                new_sentence += ch
        if flag:
            raise ValueError("Unsupported mode : {0}".format(sentence))

    elif mode == 'spelling':
        flag = True

        for ch in sentence:
            if ch == '(':
                continue
            if ch == ')':
                if flag is True:
                    flag = False
                    continue
                else:
                    flag = True
                    continue
            if ch != ')' and flag is True:
                new_sentence += ch

    else:
        raise ValueError("Unsupported mode : {0}".format(mode))

    return new_sentence


def special_filter(sentence, mode='phonetic', replace=None):
    SENTENCE_MARK = ['?', '!', '.']
    NOISE = ['o', 'n', 'u', 'b', 'l']
    EXCEPT = ['/', '+', '*', '-', '@', '$', '^', '&', '[', ']', '=', ':', ';', ','] + SENTENCE_MARK

    new_sentence = str()
    for idx, ch in enumerate(sentence):
        if ch not in SENTENCE_MARK:
            if idx + 1 < len(sentence) and ch in NOISE and sentence[idx + 1] == '/':
                continue

        if ch == '#':
            new_sentence += '샾'

        elif ch == '%':
            if mode == 'phonetic':
                new_sentence += replace
            elif mode == 'spelling':
                new_sentence += '%'

        elif ch not in EXCEPT:
            new_sentence += ch

    pattern = re.compile(r'\s\s+')
    new_sentence = re.sub(pattern, ' ', new_sentence.strip())
    return new_sentence

def sentence_filter(raw_sentence, mode, replace=None):
    return special_filter(bracket_filter(raw_sentence, mode), mode, replace)

def preprocess(raw_sentence, mode='phonetic', audio_num=0):
    percent_files = {
        '087797': '퍼센트',
        '215401': '퍼센트',
        '284574': '퍼센트',
        '397184': '퍼센트',
        '501006': '프로',
        '502173': '프로',
        '542363': '프로',
        '581483': '퍼센트'
    }

    if audio_num in percent_files.keys():
        new_sentence = sentence_filter(raw_sentence, mode, percent_files[audio_num])
    else:
        new_sentence = sentence_filter(raw_sentence, mode=mode)
    return new_sentence

# 학습시작

In [23]:
if __name__ == "__main__":
    cli_main()

RuntimeError: 